# GNN Shift Prediction Workflow

Dieses Notebook ermöglicht:
1. **Training** eines heterogenen GNN-Modells zur Shift-Vorhersage
2. **Vorhersage** mit einem trainierten Modell
3. **Erklärung** von Vorhersagen mittels GNNExplainer

---

## Setup & Imports

In [25]:
import sys
import os
from pathlib import Path

# Projektpfade einrichten
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
EXPLAINER_DIR = SCRIPTS_DIR / "explainer"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# Zentrale In-Dim-Definition (Single Source of Truth)
IN_DIM_DICT = {"H": 33, "C": 39, "Others": 16}

# Pfade zum sys.path hinzufügen
for path in [str(SCRIPTS_DIR), str(PROJECT_ROOT), str(EXPLAINER_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Projekt-Root: {PROJECT_ROOT}")
print(f"Scripts-Ordner: {SCRIPTS_DIR}")
print(f"Daten-Ordner: {DATA_DIR}")
print(f"Modelle-Ordner: {MODELS_DIR}")



Projekt-Root: /Users/sophiaberg/gnn4nmr-7
Scripts-Ordner: /Users/sophiaberg/gnn4nmr-7/scripts
Daten-Ordner: /Users/sophiaberg/gnn4nmr-7/data
Modelle-Ordner: /Users/sophiaberg/gnn4nmr-7/models


In [26]:
import random
import pickle
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Projekt-Module
from model import HeteroGNNModel
from dataloader import create_dataloaders, ShiftDataset
from train import train_model, extract_training_config

# Für GNNExplainer
from torch_geometric.explain import Explainer, HeteroExplanation
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import ModelConfig, ModelMode, ModelReturnType, ModelTaskLevel

from explainer_utils import (
    NodeTypeRegressionWrapper,
    build_dataset,
    ensure_dir,
    get_device,
    heterodata_to_dicts,
    load_config,
    load_stats,
    load_trained_model,
    summarize_incident_edges,
    summarize_node_mask,
    validate_indices,
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.9.0
CUDA verfügbar: False


---
## 1. Modell Training

Trainiere ein heterogenes GNN-Modell zur Vorhersage von chemischen Shifts.

### 1.1 Training Parameter

In [27]:
# === WandB Konfiguration ===
use_wandb_widget = widgets.Checkbox(
    value=True,
    description='WandB verwenden',
    style={'description_width': 'initial'}
)

wandb_project_widget = widgets.Text(
    value='ground truth',
    description='WandB Projekt:',
    style={'description_width': 'initial'}
)

# === Grundlegende Parameter ===
seed_widget = widgets.IntText(
    value=0,
    description='Random Seed:',
    style={'description_width': 'initial'}
)

batch_size_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=32,
    description='Batch Size:',
    style={'description_width': 'initial'}
)

num_epochs_widget = widgets.IntSlider(
    value=80,
    min=10,
    max=500,
    step=10,
    description='Epochen:',
    style={'description_width': 'initial'}
)

lr_widget = widgets.FloatLogSlider(
    value=2e-4,
    base=10,
    min=-5,
    max=-2,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

# === Modell-Architektur ===
hidden_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Hidden Dim:',
    style={'description_width': 'initial'}
)

out_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Output Dim:',
    style={'description_width': 'initial'}
)

num_gnn_layers_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=8,
    description='GNN Layers:',
    style={'description_width': 'initial'}
)

operator_type_widget = widgets.Dropdown(
    options=['SAGEConv', 'GCNConv', 'GATConv', 'GATv2Conv', 'GraphConv', 'NNConv', 'GINEConv', 'TransformerConv'],
    value='GraphConv',
    description='Operator:',
    style={'description_width': 'initial'}
)

# === Dropout ===
encoder_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='Encoder Dropout:',
    style={'description_width': 'initial'}
)

gnn_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='GNN Dropout:',
    style={'description_width': 'initial'}
)

# === Optimizer & Scheduler ===
optimizer_widget = widgets.Dropdown(
    options=['Adam', 'SGD', 'AdamW'],
    value='Adam',
    description='Optimizer:',
    style={'description_width': 'initial'}
)

weight_decay_widget = widgets.FloatLogSlider(
    value=5e-5,
    base=10,
    min=-6,
    max=-2,
    step=0.1,
    description='Weight Decay:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

scheduler_factor_widget = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=0.9,
    step=0.1,
    description='Scheduler Factor:',
    style={'description_width': 'initial'}
)

scheduler_patience_widget = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    description='Scheduler Patience:',
    style={'description_width': 'initial'}
)

# === Loss Weights ===
loss_weight_H_widget = widgets.FloatSlider(
    value=10.0,
    min=1.0,
    max=20.0,
    step=1.0,
    description='Loss Weight H:',
    style={'description_width': 'initial'}
)

loss_weight_C_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Loss Weight C:',
    style={'description_width': 'initial'}
)

# === Normalisierung ===
normalize_nodes_widget = widgets.Checkbox(
    value=True,
    description='Node Features normalisieren',
    style={'description_width': 'initial'}
)

normalize_edges_widget = widgets.Checkbox(
    value=True,
    description='Edge Features normalisieren',
    style={'description_width': 'initial'}
)

# === Split Ratio ===
train_ratio_widget = widgets.FloatSlider(
    value=0.7,
    min=0.5,
    max=0.95,
    step=0.025,
    description='Train Ratio:',
    style={'description_width': 'initial'}
)

val_ratio_widget = widgets.FloatSlider(
    value=0.15,
    min=0.0,
    max=0.3,
    step=0.025,
    description='Val Ratio:',
    style={'description_width': 'initial'}
)

# === Output ===
output_predictions_widget = widgets.Checkbox(
    value=False,
    description='Detaillierte Vorhersagen speichern',
    style={'description_width': 'initial'}
)

output_dir_widget = widgets.Text(
    value='results',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout erstellen
wandb_box = widgets.VBox([
    widgets.HTML('<h4>WandB Konfiguration</h4>'),
    use_wandb_widget,
    wandb_project_widget
])

basic_box = widgets.VBox([
    widgets.HTML('<h4>Grundeinstellungen</h4>'),
    seed_widget,
    batch_size_widget,
    num_epochs_widget,
    lr_widget
])

model_box = widgets.VBox([
    widgets.HTML('<h4>Modell-Architektur</h4>'),
    hidden_dim_widget,
    out_dim_widget,
    num_gnn_layers_widget,
    operator_type_widget,
    encoder_dropout_widget,
    gnn_dropout_widget
])

optim_box = widgets.VBox([
    widgets.HTML('<h4>Optimizer & Scheduler</h4>'),
    optimizer_widget,
    weight_decay_widget,
    scheduler_factor_widget,
    scheduler_patience_widget
])

loss_box = widgets.VBox([
    widgets.HTML('<h4>Loss & Normalisierung</h4>'),
    loss_weight_H_widget,
    loss_weight_C_widget,
    normalize_nodes_widget,
    normalize_edges_widget
])

split_box = widgets.VBox([
    widgets.HTML('<h4>Daten-Split & Output</h4>'),
    train_ratio_widget,
    val_ratio_widget,
    output_predictions_widget,
    output_dir_widget
])

# Alles anzeigen
left_column = widgets.VBox([wandb_box, basic_box, model_box])
right_column = widgets.VBox([optim_box, loss_box, split_box])

display(widgets.HBox([left_column, right_column]))

### 1.2 Training starten

In [28]:
def build_config_from_widgets():
    """Erstellt ein Config-Objekt aus den Widget-Werten."""
    class Config:
        pass
    
    config = Config()
    
    # Grundeinstellungen
    config.seed = seed_widget.value
    config.batch_size = batch_size_widget.value
    config.num_epochs = num_epochs_widget.value
    config.lr = lr_widget.value
    
    # Modell-Architektur
    config.hidden_dim = hidden_dim_widget.value
    config.out_dim = out_dim_widget.value
    config.num_gnn_layers = num_gnn_layers_widget.value
    config.operator_type = operator_type_widget.value
    config.encoder_dropout = encoder_dropout_widget.value
    config.gnnlayer_dropout = gnn_dropout_widget.value
    
    # Operator kwargs
    config.operator_kwargs = {}
    if config.operator_type in ['GATConv', 'GATv2Conv']:
        config.operator_kwargs['add_self_loops'] = False
    
    # Optimizer & Scheduler
    config.optimizer = optimizer_widget.value
    config.weight_decay = weight_decay_widget.value
    config.scheduler_factor = scheduler_factor_widget.value
    config.scheduler_patience = scheduler_patience_widget.value
    
    # Loss
    config.loss_weight_H = loss_weight_H_widget.value
    config.loss_weight_C = loss_weight_C_widget.value
    
    # Normalisierung
    config.normalize_node_features = normalize_nodes_widget.value
    config.normalize_edge_features = normalize_edges_widget.value
    
    # Split
    test_ratio = 1.0 - train_ratio_widget.value - val_ratio_widget.value
    config.split_ratio = (train_ratio_widget.value, val_ratio_widget.value, test_ratio)
    
    # Output
    config.output_detailed_predictions = output_predictions_widget.value
    config.output_dir = output_dir_widget.value
    
    # in_dim kommt zentral aus dem Notebook
    config.in_dim_dict = dict(IN_DIM_DICT)
    
    return config


def run_training():
    """Führt das Training mit den aktuellen Widget-Einstellungen durch."""
    
    # WandB initialisieren (optional)
    if use_wandb_widget.value:
        import wandb
        wandb.init(project=wandb_project_widget.value)
        config = wandb.config
        # Widget-Werte in wandb.config übertragen
        for key, value in vars(build_config_from_widgets()).items():
            setattr(config, key, value)
    else:
        config = build_config_from_widgets()
    
    # Seed setzen
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training auf: {device}")
    
    # Dataloaders erstellen
    print("Lade Daten...")
    train_loader, val_loader, test_loader = create_dataloaders(
        batch_size=config.batch_size,
        split_ratio=config.split_ratio,
        normalize_node_features=config.normalize_node_features,
        normalize_edge_features=config.normalize_edge_features
    )
    print(f"Train: {len(train_loader)} Batches, Val: {len(val_loader)} Batches, Test: {len(test_loader)} Batches")
    
    # Modell erstellen
    model = HeteroGNNModel(
        config.in_dim_dict,
        hidden_dim=config.hidden_dim,
        out_dim=config.out_dim,
        encoder_dropout=config.encoder_dropout,
        gnnlayer_dropout=config.gnnlayer_dropout,
        num_gnn_layers=config.num_gnn_layers,
        operator_type=config.operator_type,
        operator_kwargs=config.operator_kwargs,
        edge_in_dim=10
    ).to(device)
    
    print(f"\nModell erstellt: {config.operator_type} mit {config.num_gnn_layers} Layern")
    
    # Training
    print("\nStarte Training...")
    trained_model = train_model(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        config
    )
    
    # Config speichern für spätere Verwendung
    config_path = MODELS_DIR / 'config.pkl'
    os.makedirs(MODELS_DIR, exist_ok=True)
    with open(config_path, 'wb') as f:
        cfg = extract_training_config(config)
        cfg['in_dim_dict'] = dict(config.in_dim_dict)
        pickle.dump(cfg, f)
    print(f"\nKonfiguration gespeichert: {config_path}")
    
    # WandB beenden
    if use_wandb_widget.value:
        wandb.finish()
    
    return trained_model, config


# Training-Button
train_button = widgets.Button(
    description='Training starten',
    button_style='success',
    icon='play'
)

train_output = widgets.Output()

def on_train_click(b):
    with train_output:
        clear_output()
        try:
            global trained_model, training_config
            trained_model, training_config = run_training()
            print("\n✅ Training abgeschlossen!")
        except Exception as e:
            print(f"\n❌ Fehler beim Training: {e}")
            raise

train_button.on_click(on_train_click)

display(train_button)
display(train_output)

Button(button_style='success', description='Training starten', icon='play', style=ButtonStyle())

Output()

---
## 2. Vorhersagen mit trainiertem Modell

Lade ein trainiertes Modell und erstelle Vorhersagen für neue Daten.

### 2.1 Vorhersage Parameter

In [29]:
# Verfügbare Modelle und Daten auflisten
def list_files(directory, extension):
    """Listet Dateien mit bestimmter Endung in einem Verzeichnis."""
    path = Path(directory)
    if path.exists():
        return [f.name for f in path.glob(f'*{extension}')]
    return []

# Widgets für Vorhersage
model_files = list_files(MODELS_DIR, '.pt')
data_files = list_files(DATA_DIR, '.pkl')

pred_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

pred_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

pred_output_widget = widgets.Text(
    value='predictions.csv',
    description='Output Datei:',
    style={'description_width': 'initial'}
)

# Refresh Button
refresh_button = widgets.Button(
    description='Dateien aktualisieren',
    button_style='info',
    icon='refresh'
)

def on_refresh_click(b):
    model_files = list_files(MODELS_DIR, '.pt')
    data_files = list_files(DATA_DIR, '.pkl')
    pred_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    pred_data_widget.options = data_files if data_files else ['Keine Daten gefunden']
    # Auch für Explainer aktualisieren
    exp_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    exp_data_widget.options = data_files if data_files else ['Keine Daten gefunden']

refresh_button.on_click(on_refresh_click)

display(widgets.VBox([
    widgets.HTML('<h4>Vorhersage Einstellungen</h4>'),
    refresh_button,
    pred_model_widget,
    pred_data_widget,
    pred_output_widget
]))

### 2.2 Vorhersagen erstellen

In [30]:
def run_prediction():
    """Führt Vorhersagen mit dem ausgewählten Modell durch."""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Pfade
    model_path = MODELS_DIR / pred_model_widget.value
    data_path = DATA_DIR / pred_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Überprüfungen
    for path, name in [(model_path, 'Modell'), (data_path, 'Daten'), 
                       (config_path, 'Config'), (norm_stats_path, 'Norm Stats'),
                       (edge_stats_path, 'Edge Stats')]:
        if not path.exists():
            raise FileNotFoundError(f"{name} nicht gefunden: {path}")
    
    # Config laden
    with open(config_path, 'rb') as f:
        config = pickle.load(f)
    # NEU: dict-sicher machen
    if not isinstance(config, dict):
        config = vars(config) if hasattr(config, '__dict__') else dict(config)

    # Stats laden
    with open(norm_stats_path, 'rb') as f:
        norm_stats = pickle.load(f)
    with open(edge_stats_path, 'rb') as f:
        edge_stats = pickle.load(f)
    
    # Dataset erstellen
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=pred_data_widget.value,
        # NEU: dict.get statt Attribute
        normalize_node_features=config.get('normalize_node_features', True),
        normalize_edge_features=config.get('normalize_edge_features', True),
        norm_stats=norm_stats,
        **edge_stats
    )
    
    # Modell erstellen und laden
    in_dim_dict = config.get('in_dim_dict', IN_DIM_DICT)
    
    operator_kwargs = {}
    operator_type = config.get('operator_type', 'GATv2Conv')  # NEU
    if operator_type in ['GATConv', 'GATv2Conv']:
        operator_kwargs['add_self_loops'] = False
    
    model = HeteroGNNModel(
        in_dim_dict,
        hidden_dim=config.get('hidden_dim', 128),          # NEU
        out_dim=config.get('out_dim', 1),                  # NEU
        encoder_dropout=config.get('encoder_dropout', 0.0),# NEU
        gnnlayer_dropout=config.get('gnnlayer_dropout', 0.0),# NEU
        num_gnn_layers=config.get('num_gnn_layers', 3),    # NEU
        operator_type=operator_type,
        operator_kwargs=operator_kwargs,
        edge_in_dim=10
    )
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"Modell geladen: {pred_model_widget.value}")
    print(f"Daten: {pred_data_widget.value} ({len(dataset)} Graphen)")
    
    results = []
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            nx_g = dataset.nx_graphs[idx]
            compound = nx_g.graph.get("compound", f"unknown_{idx}")
            structure = nx_g.graph.get("structure", "unknown")
            
            data = dataset[idx].to(device)
            
            x_dict = {ntype: data[ntype].x for ntype in data.node_types}
            edge_index_dict = {}
            edge_attr_dict = {}
            for store in data.edge_stores:
                src, rel, dst = store._key
                edge_index_dict[(src, rel, dst)] = store.edge_index
                edge_attr_dict[(src, rel, dst)] = store.edge_attr
            
            out_dict = model(x_dict, edge_index_dict, edge_attr_dict)
            
            # Knoten sammeln
            h_nodes, c_nodes = [], []
            for node in nx_g.nodes():
                element = nx_g.nodes[node]["element"]
                if element == "H":
                    h_nodes.append(node)
                elif element == "C":
                    c_nodes.append(node)
            
            # H-Vorhersagen
            if 'H' in out_dict and out_dict['H'] is not None:
                for i, node in enumerate(h_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['H'][i].item() if i < out_dict['H'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'H',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
            
            # C-Vorhersagen
            if 'C' in out_dict and out_dict['C'] is not None:
                for i, node in enumerate(c_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['C'][i].item() if i < out_dict['C'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'C',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
    
    df = pd.DataFrame(results)
    output_path = PROJECT_ROOT / pred_output_widget.value
    df.to_csv(output_path, index=False)
    
    print(f"\n✅ {len(results)} Vorhersagen gespeichert: {output_path}")
    
    return df


# Prediction Button
predict_button = widgets.Button(
    description='Vorhersagen erstellen',
    button_style='primary',
    icon='calculator'
)

predict_output = widgets.Output()

def on_predict_click(b):
    with predict_output:
        clear_output()
        try:
            global predictions_df
            predictions_df = run_prediction()
            print("\nVorschau der Ergebnisse:")
            display(predictions_df.head(10))
        except Exception as e:
            print(f"\n❌ Fehler bei Vorhersage: {e}")
            raise

predict_button.on_click(on_predict_click)

display(predict_button)
display(predict_output)

Button(button_style='primary', description='Vorhersagen erstellen', icon='calculator', style=ButtonStyle())

Output()

---
## 3. GNNExplainer - Erklärungen generieren

Erkläre Vorhersagen für einzelne Knoten mittels GNNExplainer.

### 3.1 Explainer Parameter

In [31]:
# Explainer Defaults & Helper
GNNEXPLAINER_FALLBACK_COEFFS = {
    'edge_size': 0.005,
    'edge_ent': 1.0,
    'node_feat_size': 1.0,
    'node_feat_ent': 0.1,
}


def get_current_gnnexplainer_default_coeffs():
    """Liest die aktuellen GNNExplainer-Default-Coeffs aus der installierten PyG-Version."""
    defaults = dict(GNNEXPLAINER_FALLBACK_COEFFS)
    try:
        algo = GNNExplainer(epochs=200, lr=0.01)
        coeffs = getattr(algo, 'coeffs', None)
        if isinstance(coeffs, dict):
            for key in defaults:
                if key in coeffs:
                    defaults[key] = float(coeffs[key])
    except Exception as exc:
        print(f"Warnung: Konnte GNNExplainer-Default-Coeffs nicht introspektieren ({exc}). Nutze Fallback.")
    return defaults


def get_gnnexplainer_coeffs_from_widgets():
    return {
        'edge_size': float(exp_edge_size_coeff_widget.value),
        'edge_ent': float(exp_edge_ent_coeff_widget.value),
        'node_feat_size': float(exp_node_feat_size_coeff_widget.value),
        'node_feat_ent': float(exp_node_feat_ent_coeff_widget.value),
    }


def build_gnnexplainer_algorithm(epochs, lr, use_custom=None, coeffs=None):
    if use_custom is None:
        use_custom = bool(exp_use_custom_coeffs_widget.value)

    if not use_custom:
        return GNNExplainer(epochs=epochs, lr=lr), None

    coeffs_dict = dict(coeffs) if coeffs is not None else get_gnnexplainer_coeffs_from_widgets()

    try:
        return GNNExplainer(epochs=epochs, lr=lr, coeffs=coeffs_dict), coeffs_dict
    except TypeError:
        pass

    try:
        return GNNExplainer(epochs=epochs, lr=lr, **coeffs_dict), coeffs_dict
    except TypeError:
        pass

    algo = GNNExplainer(epochs=epochs, lr=lr)
    algo_coeffs = getattr(algo, 'coeffs', None)
    if isinstance(algo_coeffs, dict):
        algo_coeffs.update(coeffs_dict)
        return algo, coeffs_dict

    raise TypeError(
        'Custom coeffs konnten nicht auf GNNExplainer angewendet werden. '
        'Bitte torch_geometric-Version oder Coeff-Namen pr?fen.'
    )


_gnnexplainer_default_coeffs = get_current_gnnexplainer_default_coeffs()

# Explainer Widgets
exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

exp_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

exp_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

exp_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

exp_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Epochen:',
    style={'description_width': 'initial'}
)

exp_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,
    max=-1,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.3f'
)

exp_use_custom_coeffs_widget = widgets.Checkbox(
    value=False,
    description='Custom Coeffs aktivieren',
    style={'description_width': 'initial'}
)

exp_edge_size_coeff_widget = widgets.FloatLogSlider(
    value=float(_gnnexplainer_default_coeffs['edge_size']),
    base=10,
    min=-6,
    max=0,
    step=0.1,
    description='Coeff edge_size:',
    style={'description_width': 'initial'},
    readout_format='.6f'
)

exp_edge_ent_coeff_widget = widgets.FloatLogSlider(
    value=float(_gnnexplainer_default_coeffs['edge_ent']),
    base=10,
    min=-6,
    max=0,
    step=0.1,
    description='Coeff edge_ent:',
    style={'description_width': 'initial'},
    readout_format='.6f'
)

exp_node_feat_size_coeff_widget = widgets.FloatLogSlider(
    value=float(_gnnexplainer_default_coeffs['node_feat_size']),
    base=10,
    min=-6,
    max=0,
    step=0.1,
    description='Coeff node_feat_size:',
    style={'description_width': 'initial'},
    readout_format='.6f'
)

exp_node_feat_ent_coeff_widget = widgets.FloatLogSlider(
    value=float(_gnnexplainer_default_coeffs['node_feat_ent']),
    base=10,
    min=-6,
    max=0,
    step=0.1,
    description='Coeff node_feat_ent:',
    style={'description_width': 'initial'},
    readout_format='.6f'
)

exp_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'}
)

exp_topk_features_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Features:',
    style={'description_width': 'initial'}
)

exp_topk_edges_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Edges:',
    style={'description_width': 'initial'}
)

exp_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout
exp_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    exp_model_widget,
    exp_data_widget,
    exp_graph_idx_widget,
    exp_node_type_widget,
    exp_node_idx_widget,
])

exp_right = widgets.VBox([
    widgets.HTML('<h4>Explainer Einstellungen</h4>'),
    exp_epochs_widget,
    exp_lr_widget,
    exp_use_custom_coeffs_widget,
    exp_edge_size_coeff_widget,
    exp_edge_ent_coeff_widget,
    exp_node_feat_size_coeff_widget,
    exp_node_feat_ent_coeff_widget,
    exp_type_widget,
    exp_topk_features_widget,
    exp_topk_edges_widget,
    exp_output_dir_widget,
])

display(widgets.HBox([exp_left, exp_right]))


### 3.2 Erklärung generieren

In [32]:
def run_explanation(explain_all_nodes=False):
    '''Generiert eine GNNExplainer Erklärung für den ausgewählten Knoten oder alle Knoten.'''
    
    device = get_device(None)
    
    # Pfade
    model_path = MODELS_DIR / exp_model_widget.value
    data_path = DATA_DIR / exp_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Config und Stats laden
    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))
    
    # Dataset erstellen
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    validate_indices(len(dataset), exp_graph_idx_widget.value, 'graph_idx')
    
    # Daten laden
    data = dataset[exp_graph_idx_widget.value].to(device)
    x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)
    
    # Modell laden
    base_model = load_trained_model(str(model_path), config, device)

    use_custom_coeffs = bool(exp_use_custom_coeffs_widget.value)
    selected_coeffs = get_gnnexplainer_coeffs_from_widgets() if use_custom_coeffs else None
    
    # Wenn explain_all_nodes, sammle alle Erklärungen für alle Node-Typen
    if explain_all_nodes:
        print("Generiere Erklärungen für ALLE Nodes...")
        all_summaries = []
        node_types_to_explain = ['H', 'C', 'Others']
        
        # Sammle alle important_edges aus allen Nodes
        combined_important_edges = []
        
        for node_type in node_types_to_explain:
            if node_type not in x_dict:
                continue
            
            num_nodes = x_dict[node_type].size(0)
            print(f"  Erkläre {num_nodes} Nodes vom Typ {node_type}...")
            
            wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
            target = None
            target_tensor = y_dict.get(node_type)
            
            if target_tensor is not None and exp_type_widget.value == 'phenomenon':
                target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
                target = target_tensor
            
            model_config = ModelConfig(
                mode=ModelMode.regression,
                task_level=ModelTaskLevel.node,
                return_type=ModelReturnType.raw,
            )
            
            algorithm, used_coeffs = build_gnnexplainer_algorithm(
                epochs=exp_epochs_widget.value,
                lr=exp_lr_widget.value,
                use_custom=use_custom_coeffs,
                coeffs=selected_coeffs,
            )
            explainer = Explainer(
                model=wrapped_model,
                algorithm=algorithm,
                explanation_type=exp_type_widget.value,
                model_config=model_config,
                node_mask_type='attributes',
                edge_mask_type='object',
            )
            
            for node_idx in range(num_nodes):
                try:
                    if target is not None and torch.isnan(target[node_idx]):
                        continue
                    
                    explanation = explainer(
                        x_dict,
                        edge_index_dict,
                        edge_attr_dict=edge_attr_dict,
                        target=target,
                        index=node_idx,
                    )
                    
                    # Edge summary für diesen Node
                    edge_summary = []
                    for edge_type, mask in explanation.edge_mask_dict.items():
                        if mask is None:
                            continue
                        edge_index = edge_index_dict.get(edge_type)
                        if edge_index is None:
                            continue
                        mask_vals = mask.view(-1).detach().cpu()
                        rows = edge_index[0].detach().cpu()
                        cols = edge_index[1].detach().cpu()
                        for edge_pos, importance in enumerate(mask_vals):
                            edge_summary.append({
                                'edge_type': edge_type,
                                'edge_position': edge_pos,
                                'importance': float(importance),
                                'src_index': int(rows[edge_pos]),
                                'dst_index': int(cols[edge_pos]),
                            })
                    
                    edge_summary.sort(key=lambda item: item['importance'], reverse=True)
                    
                    # Füge alle Edges zur kombinierten Liste hinzu
                    combined_important_edges.extend(edge_summary)
                    
                    summary = {
                        'graph_idx': exp_graph_idx_widget.value,
                        'node_type': node_type,
                        'node_idx': node_idx,
                        'important_edges': edge_summary,
                        'gnnexplainer_epochs': exp_epochs_widget.value,
                        'gnnexplainer_lr': exp_lr_widget.value,
                        'gnnexplainer_coeffs': selected_coeffs,
                    }
                    
                    all_summaries.append(summary)
                    
                except Exception as e:
                    print(f"  Fehler bei {node_type}[{node_idx}]: {e}")
                    continue
        
        print(f"✅ {len(all_summaries)} Erklärungen generiert")
        
        # Erstelle eine kombinierte Summary mit allen Edges
        combined_summary = {
            'graph_idx': exp_graph_idx_widget.value,
            'node_type': 'ALL',
            'node_idx': -1,
            'important_edges': combined_important_edges,
            'gnnexplainer_epochs': exp_epochs_widget.value,
            'gnnexplainer_lr': exp_lr_widget.value,
            'gnnexplainer_coeffs': selected_coeffs,
        }
        
        # Speichere global
        global all_node_summaries
        all_node_summaries = all_summaries
        
        # Speichere auch ein dummy explanation object
        global explanation_obj
        if all_summaries:
            # Nutze die erste Node-Explanation als Basis
            wrapped_model = NodeTypeRegressionWrapper(base_model, all_summaries[0]['node_type'])
            target_tensor = y_dict.get(all_summaries[0]['node_type'])
            target = None
            if target_tensor is not None and exp_type_widget.value == 'phenomenon':
                target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
                target = target_tensor
            
            model_config = ModelConfig(
                mode=ModelMode.regression,
                task_level=ModelTaskLevel.node,
                return_type=ModelReturnType.raw,
            )
            
            algorithm, used_coeffs = build_gnnexplainer_algorithm(
                epochs=exp_epochs_widget.value,
                lr=exp_lr_widget.value,
                use_custom=use_custom_coeffs,
                coeffs=selected_coeffs,
            )
            explainer = Explainer(
                model=wrapped_model,
                algorithm=algorithm,
                explanation_type=exp_type_widget.value,
                model_config=model_config,
                node_mask_type='attributes',
                edge_mask_type='object',
            )
            
            explanation_obj = explainer(
                x_dict,
                edge_index_dict,
                edge_attr_dict=edge_attr_dict,
                target=target,
                index=all_summaries[0]['node_idx'],
            )
        
        return combined_summary, explanation_obj
    
    # Einzelner Node (originales Verhalten)
    wrapped_model = NodeTypeRegressionWrapper(base_model, exp_node_type_widget.value)
    
    # Target vorbereiten
    target = None
    target_tensor = y_dict.get(exp_node_type_widget.value)
    
    if target_tensor is not None:
        target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
        validate_indices(target_tensor.size(0), exp_node_idx_widget.value, 'node_idx')
        if exp_type_widget.value == 'phenomenon':
            if torch.isnan(target_tensor[exp_node_idx_widget.value]):
                raise ValueError(
                    f'Target für Knoten {exp_node_idx_widget.value} ist NaN. '
                    "Wähle einen anderen Knoten oder nutze 'model' als Explanation Type."
                )
            target = target_tensor
    else:
        validate_indices(x_dict[exp_node_type_widget.value].size(0), exp_node_idx_widget.value, 'node_idx')
        if exp_type_widget.value == 'phenomenon':
            raise ValueError(
                f'Keine Targets für Node Type {exp_node_type_widget.value}; '
                'kann phenomenon nicht erklären.'
            )
    
    # Explainer konfigurieren
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    
    algorithm, used_coeffs = build_gnnexplainer_algorithm(
        epochs=exp_epochs_widget.value,
        lr=exp_lr_widget.value,
        use_custom=use_custom_coeffs,
        coeffs=selected_coeffs,
    )
    explainer = Explainer(
        model=wrapped_model,
        algorithm=algorithm,
        explanation_type=exp_type_widget.value,
        model_config=model_config,
        node_mask_type='attributes',
        edge_mask_type='object',
    )
    
    print(f'Generiere Erklärung für {exp_node_type_widget.value}[{exp_node_idx_widget.value}]...')
    
    # Erklärung generieren
    explanation = explainer(
        x_dict,
        edge_index_dict,
        edge_attr_dict=edge_attr_dict,
        target=target,
        index=exp_node_idx_widget.value,
    )
    
    if not isinstance(explanation, HeteroExplanation):
        raise TypeError(
            f'Erwartete HeteroExplanation, erhielt {type(explanation)}. '
            'Stelle sicher, dass eine aktuelle torch_geometric Version verwendet wird.'
        )
    
    # Vorhersage holen
    with torch.no_grad():
        predictions = base_model(x_dict, edge_index_dict, edge_attr_dict)
        node_prediction = float(predictions[exp_node_type_widget.value][exp_node_idx_widget.value].item())
    
    # Zusammenfassungen erstellen
    feature_summary = summarize_node_mask(
        explanation.node_mask_dict,
        exp_node_type_widget.value,
        exp_node_idx_widget.value,
        top_k=max(exp_topk_features_widget.value, 0),
    )
    
    # Alle Kanten aus der Edge-Mask sammeln (keine Filter), danach nach Importance sortieren
    edge_summary = []
    for edge_type, mask in explanation.edge_mask_dict.items():
        if mask is None:
            continue
        edge_index = edge_index_dict.get(edge_type)
        if edge_index is None:
            continue
        mask_vals = mask.view(-1).detach().cpu()
        rows = edge_index[0].detach().cpu()
        cols = edge_index[1].detach().cpu()
        for edge_pos, importance in enumerate(mask_vals):
            edge_summary.append(
                {
                    'edge_type': edge_type,
                    'edge_position': edge_pos,
                    'importance': float(importance),
                    'src_index': int(rows[edge_pos]),
                    'dst_index': int(cols[edge_pos]),
                }
            )
    edge_summary.sort(key=lambda item: item['importance'], reverse=True)
    
    # ASCII-Tabelle zur schnellen Sichtkontrolle (gekürzt auf 40 Zeilen)
    edge_table_rows: List[Dict[str, Any]] = []
    for entry in edge_summary:
        src_type, _rel_type, dst_type = entry['edge_type']
        src_idx = entry.get('src_index')
        dst_idx = entry.get('dst_index')
        edge_table_rows.append(
            {
                'src_label': f'{src_type}[{src_idx}]',
                'dst_label': f'{dst_type}[{dst_idx}]',
                'edge_pos': entry.get('edge_position'),
                'importance': entry.get('importance'),
            }
        )
    max_rows_print = 40
    col_specs = [
        ('src_label', 14),
        ('dst_label', 14),
        ('edge_pos', 9),
        ('importance', 12),
    ]
    def fmt_row(row_dict):
        cells = []
        for key, width in col_specs:
            val = row_dict[key]
            if key == 'importance':
                cells.append(f"{val:>{width}.6f}")
            else:
                cells.append(f"{str(val):<{width}}")
        return ' '.join(cells)
    header = ' '.join(f"{name:<{width}}" for name, width in col_specs)
    divider = '-' * len(header)
    table_lines = ['EDGE TABLE (top nach Importance, gekürzt)', header, divider]
    for row in edge_table_rows[:max_rows_print]:
        table_lines.append(fmt_row(row))
    if len(edge_table_rows) > max_rows_print:
        table_lines.append(f"... ({len(edge_table_rows) - max_rows_print} weitere Zeilen)")
    print('\n'.join(table_lines))
    
    target_value = float(target[exp_node_idx_widget.value].item()) if target is not None else None
    
    summary = {
        'graph_idx': exp_graph_idx_widget.value,
        'node_type': exp_node_type_widget.value,
        'node_idx': exp_node_idx_widget.value,
        'prediction': node_prediction,
        'target': target_value,
        'top_features': feature_summary,
        'important_edges': edge_summary,
        'gnnexplainer_epochs': exp_epochs_widget.value,
        'gnnexplainer_lr': exp_lr_widget.value,
        'gnnexplainer_coeffs': selected_coeffs,
    }
    
    # Speichern (.pt + JSON mit Summary/Edges)
    output_dir = PROJECT_ROOT / exp_output_dir_widget.value
    ensure_dir(str(output_dir))
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base_name = f'gnn_explainer_{exp_node_type_widget.value}_n{exp_node_idx_widget.value}_g{exp_graph_idx_widget.value}_{timestamp}'
    
    torch.save(explanation, output_dir / f'{base_name}.pt')
    with open(output_dir / f'{base_name}.json', 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)
    
    print(f"\n✅ Erklärung gespeichert in: {output_dir}")
    
    return summary, explanation

# Explain Button
explain_button = widgets.Button(
    description='Erklärung generieren',
    button_style='warning',
    icon='lightbulb'
    )

explain_output = widgets.Output()

def on_explain_click(b):
    with explain_output:
        clear_output()
        try:
            global explanation_summary, explanation_obj
            explanation_summary, explanation_obj = run_explanation()
            print('\n' + '='*50)
            print('ERKLÄRUNGSZUSAMMENFASSUNG')
            print('='*50)
            print(json.dumps(explanation_summary, indent=2))
        except Exception as e:
            print(f'\n❌ Fehler bei Erklärung: {e}')
            raise

explain_button.on_click(on_explain_click)

display(explain_button)
display(explain_output)


Button(button_style='warning', description='Erklärung generieren', icon='lightbulb', style=ButtonStyle())

Output()

### Visualisierung

In [33]:
"""
# Generate explanations for ALL nodes with ALL edges (wie in Zelle 19)
from torch_geometric.explain import Explainer

# Use graph index selected in Cell 33
graph_idx = int(exp_graph_idx_widget.value)
print(f"Using graph index: {graph_idx}")

# Load selected graph data
data = dataset[graph_idx]

# Build edge_index_dict for the selected graph
edge_index_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_index_dict[key] = store.edge_index

# Build atom_index_dict from NetworkX graph for SDF mapping
nx_g = dataset.nx_graphs[graph_idx]
atom_index_dict = {"H": [], "C": [], "Others": []}
for node in nx_g.nodes():
    element = nx_g.nodes[node]["element"]
    if element in ["H", "C"]:
        atom_index_dict[element].append(node)
    else:
        atom_index_dict["Others"].append(node)

# Convert to tensors for indexing with local indices
for ntype in atom_index_dict:
    if atom_index_dict[ntype]:
        atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
    else:
        atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

# Prepare feature and edge attribute dicts from selected data
x_dict = {ntype: data[ntype].x for ntype in data.node_types}
edge_attr_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_attr_dict[key] = store.edge_attr

y_dict = {}
for ntype in data.node_types:
    y_val = getattr(data[ntype], 'y', None)
    y_dict[ntype] = y_val

# Load model
in_dim_dict = config.get('in_dim_dict', IN_DIM_DICT)
operator_kwargs = {}
operator_type = config.get('operator_type', 'GATv2Conv')
if operator_type in ['GATConv', 'GATv2Conv']:
    operator_kwargs['add_self_loops'] = False

base_model = HeteroGNNModel(
    in_dim_dict,
    hidden_dim=config.get('hidden_dim', 128),
    out_dim=config.get('out_dim', 1),
    encoder_dropout=config.get('encoder_dropout', 0.0),
    gnnlayer_dropout=config.get('gnnlayer_dropout', 0.0),
    num_gnn_layers=config.get('num_gnn_layers', 3),
    operator_type=operator_type,
    operator_kwargs=operator_kwargs,
    edge_in_dim=10
)
base_model.load_state_dict(torch.load(str(MODELS_DIR / exp_model_widget.value), map_location='cpu'))
base_model.eval()

use_custom_coeffs = bool(exp_use_custom_coeffs_widget.value)
selected_coeffs = get_gnnexplainer_coeffs_from_widgets() if use_custom_coeffs else None

# Generate explanations for ALL nodes
print("=" * 80)
print("Generiere Erklärungen für ALLE Nodes...")
print("=" * 80)
all_node_summaries = []
node_types_to_explain = ['H', 'C', 'Others']

# Store edge importances per node (SDF index)
edges_per_node = {}

for node_type in node_types_to_explain:
    if node_type not in x_dict:
        continue
    
    num_nodes = x_dict[node_type].size(0)
    print(f"\n{'='*80}")
    print(f"Node Type: {node_type} ({num_nodes} nodes)")
    print(f"{'='*80}")
    
    wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
    target = None
    target_tensor = y_dict.get(node_type)
    
    if target_tensor is not None and exp_type_widget.value == 'phenomenon':
        target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
        target = target_tensor
    
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    
    algorithm, used_coeffs = build_gnnexplainer_algorithm(
        epochs=exp_epochs_widget.value,
        lr=exp_lr_widget.value,
        use_custom=use_custom_coeffs,
        coeffs=selected_coeffs,
    )
    explainer = Explainer(
        model=wrapped_model,
        algorithm=algorithm,
        explanation_type=exp_type_widget.value,
        model_config=model_config,
        node_mask_type='attributes',
        edge_mask_type='object',
    )
    
    for node_idx in range(num_nodes):
        try:
            if target is not None and torch.isnan(target[node_idx]):
                continue
            
            explanation = explainer(
                x_dict,
                edge_index_dict,
                edge_attr_dict=edge_attr_dict,
                target=target,
                index=node_idx,
            )
            
            # Collect ALL edges with their importances
            edge_summary = []
            for edge_type, mask in explanation.edge_mask_dict.items():
                if mask is None:
                    continue
                edge_index = edge_index_dict.get(edge_type)
                if edge_index is None:
                    continue
                mask_vals = mask.view(-1).detach().cpu()
                rows = edge_index[0].detach().cpu()
                cols = edge_index[1].detach().cpu()
                for edge_pos, importance in enumerate(mask_vals):
                    edge_summary.append({
                        'edge_type': edge_type,
                        'edge_position': edge_pos,
                        'importance': float(importance),
                        'src_index': int(rows[edge_pos]),
                        'dst_index': int(cols[edge_pos]),
                    })
            
            edge_summary.sort(key=lambda x: abs(x['importance']), reverse=True)
            
            # Find SDF index for this node
            sdf_idx = None
            if atom_index_dict[node_type].numel() > 0:
                for i, idx in enumerate(atom_index_dict[node_type]):
                    if i == node_idx:
                        sdf_idx = int(idx.item())
                        break
            
            if sdf_idx is not None:
                # Convert edge indices to SDF indices
                # Use dict to handle bidirectional edges: keep max importance
                edge_dict = {}
                for edge in edge_summary:
                    edge_type_tuple = edge['edge_type']
                    src_type, _, dst_type = edge_type_tuple
                    src_local = edge['src_index']
                    dst_local = edge['dst_index']
                    
                    if atom_index_dict[src_type].numel() > src_local and atom_index_dict[dst_type].numel() > dst_local:
                        src_sdf = int(atom_index_dict[src_type][src_local].item())
                        dst_sdf = int(atom_index_dict[dst_type][dst_local].item())
                        
                        # Canonical edge key (kleinerer Index zuerst, wie in Zelle 24)
                        edge_key = tuple(sorted([src_sdf, dst_sdf]))
                        
                        # Behalte Maximum bei bidirektionalen Kanten
                        if edge_key in edge_dict:
                            if abs(edge['importance']) > abs(edge_dict[edge_key]['importance']):
                                edge_dict[edge_key] = {
                                    'src': src_sdf,
                                    'dst': dst_sdf,
                                    'importance': edge['importance'],
                                }
                        else:
                            edge_dict[edge_key] = {
                                'src': src_sdf,
                                'dst': dst_sdf,
                                'importance': edge['importance'],
                            }
                
                all_edges_for_node = list(edge_dict.values())
                # Sortiere nach Importance
                all_edges_for_node.sort(key=lambda x: abs(x['importance']), reverse=True)
                
                edges_per_node[sdf_idx] = all_edges_for_node
                
                # Print summary for this node
                symbol = atoms[sdf_idx]['symbol'] if sdf_idx < len(atoms) else '?'
                print(f"\n  Node {sdf_idx} ({symbol}, {node_type}[{node_idx}]): {len(all_edges_for_node)} edges")
                
                # Show ALL edges
                if len(all_edges_for_node) > 0:
                    print(f"    Alle Edges (sortiert nach Importance):")
                    for i, edge in enumerate(all_edges_for_node):
                        src_sym = atoms[edge['src']]['symbol'] if edge['src'] < len(atoms) else '?'
                        dst_sym = atoms[edge['dst']]['symbol'] if edge['dst'] < len(atoms) else '?'
                        print(f"      {i+1:3d}. {edge['src']:2d}({src_sym}) ─ {edge['dst']:2d}({dst_sym}): {edge['importance']:+.6f}")
            
        except Exception as e:
            print(f"  ❌ Fehler bei {node_type}[{node_idx}]: {e}")
            continue

print(f"\n{'='*80}")
print(f"✓ FERTIG!")
print(f"{'='*80}")
print(f"  Erklärungen generiert: {len(edges_per_node)} nodes")
print(f"  Total atoms: {len(atoms)}")
print(f"{'='*80}")
"""

'\n# Generate explanations for ALL nodes with ALL edges (wie in Zelle 19)\nfrom torch_geometric.explain import Explainer\n\n# Use graph index selected in Cell 33\ngraph_idx = int(exp_graph_idx_widget.value)\nprint(f"Using graph index: {graph_idx}")\n\n# Load selected graph data\ndata = dataset[graph_idx]\n\n# Build edge_index_dict for the selected graph\nedge_index_dict = {}\nfor store in data.edge_stores:\n    key = store._key\n    edge_index_dict[key] = store.edge_index\n\n# Build atom_index_dict from NetworkX graph for SDF mapping\nnx_g = dataset.nx_graphs[graph_idx]\natom_index_dict = {"H": [], "C": [], "Others": []}\nfor node in nx_g.nodes():\n    element = nx_g.nodes[node]["element"]\n    if element in ["H", "C"]:\n        atom_index_dict[element].append(node)\n    else:\n        atom_index_dict["Others"].append(node)\n\n# Convert to tensors for indexing with local indices\nfor ntype in atom_index_dict:\n    if atom_index_dict[ntype]:\n        atom_index_dict[ntype] = torch.tenso

In [34]:
def load_sdf_atoms(path):
 
    atoms = []
    with open(path) as f:
        lines = f.readlines()

    if len(lines) < 5:
        return atoms

    counts_line = lines[3]
    try:
        n_atoms = int(counts_line[0:3])
    except ValueError:
        return atoms

    atom_lines = lines[4 : 4 + n_atoms]

    for idx, line in enumerate(atom_lines):
        try:
            x = float(line[0:10])
            y = float(line[10:20])
            z = float(line[20:30])
            symbol = line[31:34].strip()
        except ValueError:
            continue
        atoms.append(
            {
                "index": idx,
                "symbol": symbol,
                "x": x,
                "y": y,
                "z": z,
            }
        )
    return atoms


def calculate_all_atom_importances(nx_g, edge_index_dict, edge_masks, atom_index_dict):
    """
    Compute per-atom importance as the max absolute edge importance over all incident edges.
    Robust to mismatches between edge_index size and mask length by clamping the iteration range.
    """
    atom_importance = {}
    for ntype, atom_indices in atom_index_dict.items():
        if atom_indices.numel() == 0:
            continue
        # Collect edge types touching this node type
        relevant_edge_types = []
        for edge_type in edge_index_dict.keys():
            src_type, _, dst_type = edge_type
            if src_type == ntype or dst_type == ntype:
                relevant_edge_types.append(edge_type)
        # Walk local indices
        for local_idx in range(len(atom_indices)):
            sdf_idx = int(atom_indices[local_idx].item())
            max_importance = 0.0
            for edge_type in relevant_edge_types:
                if edge_type not in edge_masks:
                    continue
                mask = edge_masks[edge_type]
                if mask is None:
                    continue
                # Flatten mask and clamp length to number of edges
                mask = mask.view(-1)
                ei = edge_index_dict[edge_type]
                num_edges = int(ei.size(1))
                num_mask = int(mask.shape[0])
                limit = min(num_edges, num_mask)
                if limit <= 0:
                    continue
                src_type, _, dst_type = edge_type
                # Outgoing
                for edge_idx in range(limit):
                    if ei[0, edge_idx] == local_idx:
                        try:
                            max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                        except Exception:
                            pass
                # Incoming
                for edge_idx in range(limit):
                    if ei[1, edge_idx] == local_idx:
                        try:
                            max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                        except Exception:
                            pass
            atom_importance[sdf_idx] = max_importance
    return atom_importance


# Use graph index selected in Cell 33
import json
import torch
import py3Dmol
import pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

graph_idx = int(exp_graph_idx_widget.value)
print(f"Using graph index (Cell 33): {graph_idx}")

data_file = exp_data_widget.value
config_path = MODELS_DIR / "config.pkl"
norm_stats_path = MODELS_DIR / "norm_stats.pkl"
edge_stats_path = MODELS_DIR / "edge_stats.pkl"

with open(config_path, "rb") as f:
    config = pickle.load(f)
with open(norm_stats_path, "rb") as f:
    norm_stats = pickle.load(f)
with open(edge_stats_path, "rb") as f:
    edge_stats = pickle.load(f)

# Load dataset and selected graph
dataset = build_dataset(str(DATA_DIR / data_file), config, norm_stats=norm_stats, edge_stats=edge_stats)

nx_g = dataset.nx_graphs[graph_idx]
data = dataset[graph_idx]

# Build atom index dict (SDF mapping)
atom_index_dict = {"H": [], "C": [], "Others": []}
for node in nx_g.nodes():
    element = nx_g.nodes[node]["element"]
    if element in ["H", "C"]:
        atom_index_dict[element].append(node)
    else:
        atom_index_dict["Others"].append(node)

for ntype in atom_index_dict:
    if atom_index_dict[ntype]:
        atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
    else:
        atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

# Build edge index dict
edge_index_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_index_dict[key] = store.edge_index

# Gather edges and masks from global explanation if present; else derive from available data
edge_masks = explanation_obj.edge_mask_dict if 'explanation_obj' in globals() and hasattr(explanation_obj, 'edge_mask_dict') else {}

# Compute atom importances (max of incident edges)
atom_importance = calculate_all_atom_importances(nx_g, edge_index_dict, edge_masks, atom_index_dict)

# Prepare SDF
compound = nx_g.graph.get("compound", 1)
structure = nx_g.graph.get("structure", 1)
subfolder = f"{int(compound):03d}"
sdf_name = f"{subfolder}_{int(structure):02d}.sdf"
sdf_path = DATA_DIR / "orca_xyz_formate" / subfolder / sdf_name
print(f"Looking for SDF file: {sdf_path}")

if not sdf_path.exists():
    print("SDF file not found for selected graph.")
else:
    with open(str(sdf_path)) as f:
        sdf_block = f.read()
    atoms = load_sdf_atoms(str(sdf_path))

    # Global edges from masks (if available)
    seen_edges = set()
    all_edges_list = []
    for edge_type in edge_index_dict.keys():
        if edge_type not in edge_masks:
            continue
        ei = edge_index_dict[edge_type]
        mask = edge_masks[edge_type]
        if mask is None:
            continue
        mask = mask.view(-1)
        num_edges = int(ei.size(1))
        num_mask = int(mask.shape[0])
        limit = min(num_edges, num_mask)
        src_type, _, dst_type = edge_type
        for edge_idx in range(limit):
            src_local = int(ei[0, edge_idx].item())
            dst_local = int(ei[1, edge_idx].item())
            if atom_index_dict[src_type].numel() > src_local and atom_index_dict[dst_type].numel() > dst_local:
                src_sdf = int(atom_index_dict[src_type][src_local].item())
                dst_sdf = int(atom_index_dict[dst_type][dst_local].item())
                importance = float(mask[edge_idx].item())
                edge_key = tuple(sorted([src_sdf, dst_sdf])) + (edge_type,)
                if edge_key not in seen_edges:
                    seen_edges.add(edge_key)
                    all_edges_list.append({
                        'src': src_sdf,
                        'dst': dst_sdf,
                        'edge_type': f"{src_type}→{dst_type}",
                        'importance': importance,
                        'edge_idx': edge_idx
                    })
    print(f"Collected {len(all_edges_list)} unique edges from edge_mask_dict for graph {graph_idx}")

    # Build per-node colors using edges_per_node (if present)
    edges_colors_per_node = {}
    if 'edges_per_node' in globals():
        for node_idx, node_edges in edges_per_node.items():
            node_importances = [abs(edge['importance']) for edge in node_edges]
            # Normalizer
            def make_normalizer(values, default=1.0):
                if not values:
                    return lambda v: default
                vmin = min(values); vmax = max(values)
                def norm(v):
                    if vmax == vmin:
                        return default
                    return (v - vmin) / (vmax - vmin)
                return norm
            norm_func = make_normalizer(node_importances)
            # Blue->Red without green
            def importance_to_color_edges(v):
                v = max(0.0, min(1.0, v))
                r = int(v * 255); g = 0; b = int((1.0 - v) * 255)
                return f"rgb({r},{g},{b})"
            colored_edges = []
            for edge in node_edges:
                imp = edge['importance']
                norm_val = norm_func(abs(imp))
                color = importance_to_color_edges(norm_val)
                colored_edges.append({
                    'src': edge['src'],
                    'dst': edge['dst'],
                    'importance': imp,
                    'color': color
                })
            edges_colors_per_node[node_idx] = colored_edges
    edges_colors_per_node_json = json.dumps(edges_colors_per_node)
    print(f"Prepared node-specific edge colors for {len(edges_colors_per_node)} nodes (graph {graph_idx})")

    # Atom colors (optional)
    atom_colors = {}
    if atom_importance:
        vals = list(atom_importance.values())
        def make_atom_norm(vals):
            if not vals:
                return lambda x: 0.0
            m = min(vals); M = max(vals)
            if abs(M - m) < 1e-12:
                return lambda x: 0.0
            return lambda x: (x - m) / (M - m)
        norm_atoms = make_atom_norm(vals)
        def importance_to_color(v):
            if v <= 0:
                return 'blue'
            r = int(255 * v); g = 0; b = int(255 * (1 - v))
            return f'rgb({r},{g},{b})'
        for atom_idx, imp in atom_importance.items():
            v = norm_atoms(imp)
            c = importance_to_color(v)
            atom_colors[str(atom_idx)] = c
    atom_colors_json = json.dumps(atom_colors)

    # Create py3Dmol view
    view = py3Dmol.view(width=700, height=600)
    view.addModel(sdf_block, "sdf")
    view.setStyle({}, {"stick": {"color": "lightgray", "radius": 0.2}, "sphere": {"color": "lightgray", "radius": 0.4}})

    # Labels for non-C atoms
    if atoms:
        for atom in atoms:
            if atom["symbol"] != "C":
                view.addLabel(
                    atom["symbol"],
                    {
                        "position": {"x": atom["x"], "y": atom["y"], "z": atom["z"]},
                        "fontSize": 12,
                        "fontColor": "black",
                        "backgroundColor": "white",
                        "showBackground": True,
                    },
                )

    # Click behavior
    view.addStyle({}, {"clicksphere": {"radius": 0.8}})
    click_cb = f"""
    function(atom, viewer, event, container) {{
        var edgesPerNode = {edges_colors_per_node_json};
        var atomIdx = atom.index;
        if (!viewer.__selectedAtomInfo) {{ viewer.__selectedAtomInfo = {{}}; }}
        viewer.setStyle({{}}, {{stick: {{color: \"lightgray\", radius: 0.2}}, sphere: {{color: \"lightgray\", radius: 0.4}}}});
        viewer.addStyle({{index: atomIdx}}, {{sphere: {{color: \"yellow\", radius: 0.5}}}});
        var nodeEdges = edgesPerNode[atomIdx.toString()];
        if (nodeEdges && nodeEdges.length > 0) {{
            for (var i=0; i<nodeEdges.length; i++) {{
                var edge = nodeEdges[i];
                var src = edge.src; var dst = edge.dst; var color = edge.color;
                viewer.addStyle({{index: src}}, {{stick: {{color: color, radius: 0.3}}}});
                viewer.addStyle({{index: dst}}, {{stick: {{color: color, radius: 0.3}}}});
                if (src !== atomIdx) {{ viewer.addStyle({{index: src}}, {{sphere: {{color: color, radius: 0.45}}}}); }}
                if (dst !== atomIdx) {{ viewer.addStyle({{index: dst}}, {{sphere: {{color: color, radius: 0.45}}}}); }}
            }}
        }}
        if (viewer.__selectedLabel) {{ viewer.removeLabel(viewer.__selectedLabel); }}
        viewer.__selectedLabel = viewer.addLabel(\"Atom \" + atomIdx, {{ position: atom, backgroundColor: \"white\", fontColor: \"black\", fontSize: 12, showBackground: true }});
        viewer.__selectedAtomInfo.index = atomIdx;
        viewer.render();
    }}
    """
    view.setClickable({}, True, click_cb)
    view.zoomTo(); view.render(); view.show()

Using graph index (Cell 33): 0
Looking for SDF file: /Users/sophiaberg/gnn4nmr-7/data/orca_xyz_formate/001/001_00.sdf
Collected 0 unique edges from edge_mask_dict for graph 0
Prepared node-specific edge colors for 0 nodes (graph 0)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Batch Analysis GNNExplainer


In [39]:
# === GNNExplainer Batch-Analyse im vereinfachten Format pro Node-Type ===

import os
import random
import numpy as np
from datetime import datetime
from pathlib import Path

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.explain.config import (
    ModelConfig,
    ModelMode,
    ModelTaskLevel,
    ModelReturnType,
)

# ---------------------------------------------------------------------------
# Fallback feature names
# ---------------------------------------------------------------------------
try:
    feature_names_for
except NameError:
    def feature_names_for(node_type, length):
        base = get_feature_names(node_type)
        if len(base) < length:
            base = base + [f"{node_type}_extra_{i}" for i in range(len(base), length)]
        return base[:length]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _parse_graph_indices_gnn_like_ig(text, total):
    text = ''.join((text or '').split())
    if not text:
        return None
    seen = set()
    result = []
    for token in text.split(','):
        if not token:
            continue
        if '-' in token:
            try:
                a, b = token.split('-', 1)
                a, b = int(a), int(b)
            except ValueError:
                continue
            if a > b:
                a, b = b, a
            for idx in range(a, b + 1):
                if 0 <= idx < total and idx not in seen:
                    seen.add(idx)
                    result.append(idx)
        else:
            try:
                idx = int(token)
            except ValueError:
                continue
            if 0 <= idx < total and idx not in seen:
                seen.add(idx)
                result.append(idx)
    return sorted(result) if result else None


def _safe_mean(vectors):
    if not vectors:
        return []
    arr = np.stack(vectors, axis=0)
    return np.mean(arr, axis=0).tolist()


# ---------------------------------------------------------------------------
# Summary / Ranking helpers
# ---------------------------------------------------------------------------
def _print_batch_summary_gnn_simple(batch_results: dict, graph_indices: list, title: str):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    for nt, result in batch_results.items():
        print(f"\nNode Type: {nt}")
        print(f"  Analysierte Graphen: {len(graph_indices)}")
        print(f"  Erklärte Knoten / gemittelte Masken: {result.get('node_count', 0)}")
        print(f"  Features: {len(result.get('avg_importance', []))}")

        importances = result.get("avg_importance", [])
        if not importances:
            print("  (keine Daten)")
            continue

        print("  Top 5 wichtigste Features:")
        feature_names = feature_names_for(nt, len(importances))
        sorted_indices = sorted(
            range(len(importances)),
            key=lambda i: importances[i],
            reverse=True
        )
        for i in range(min(5, len(sorted_indices))):
            idx = sorted_indices[i]
            name = feature_names[idx]
            print(f"    {name}: {importances[idx]:.6f}")


def _print_joint_top_features_gnn_simple(
    batch_results: dict,
    top_k: int | None = None,
    normalize_per_type: bool = False,
):
    rows = []

    for nt, res in batch_results.items():
        imp = res.get("avg_importance", [])
        if not imp:
            continue

        imp = np.array(imp, dtype=float)
        if normalize_per_type:
            denom = float(np.sum(imp))
            if denom < 1e-12:
                continue
            scores = imp / denom
        else:
            scores = imp

        names = feature_names_for(nt, len(scores))
        for i in range(len(scores)):
            rows.append({
                "node_type": nt,
                "feature_idx": i,
                "feature_name": names[i],
                "score": float(scores[i]),
            })

    if not rows:
        print("\n[Joint Top Features] Keine Daten vorhanden.")
        return

    rows.sort(key=lambda r: r["score"], reverse=True)
    top_n = len(rows) if (top_k is None or top_k <= 0) else min(top_k, len(rows))
    title_extra = "(alle Features)" if (top_k is None or top_k <= 0) else f"(Top {top_n})"

    print("\n" + "=" * 70)
    print("JOINT FEATURE RANKING (across node types)")
    if normalize_per_type:
        print(f"Ranking basis: per-type normalized avg_importance {title_extra}")
    else:
        print(f"Ranking basis: RAW avg_importance {title_extra}")
    print("=" * 70)

    for rank, r in enumerate(rows[:top_n], start=1):
        print(
            f"{rank:>3}. {r['node_type']}: {r['feature_name']} "
            f"(idx={r['feature_idx']}) | score={r['score']:.6f}"
        )



def _build_joint_ranking_rows_gnn_simple(batch_results: dict, top_k: int | None = None):
    """Build reusable joint ranking rows from global GNN batch result for one target type."""
    rows = []
    for nt, res in batch_results.items():
        imp = res.get("avg_importance", [])
        if not imp:
            continue
        vals = np.array(imp, dtype=float)
        names = feature_names_for(nt, len(vals))
        for i in range(len(vals)):
            abs_v = float(abs(vals[i]))
            rows.append({
                "feature_key": f"{nt}:{names[i]}",
                "node_type": nt,
                "feature_name": names[i],
                "importance": abs_v,
                "value": abs_v,
            })

    rows.sort(key=lambda r: r["importance"], reverse=True)
    if top_k is not None and top_k > 0:
        rows = rows[:top_k]
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return rows


# ---------------------------------------------------------------------------
# Core runner: ONE target node type -> simple per-node-type result structure
# ---------------------------------------------------------------------------
def run_gnn_batch_analysis_simple(
    model_path,
    data_path,
    config_path,
    norm_stats_path,
    edge_stats_path,
    graph_indices,
    target_node_type,
    epochs=100,
    expl_type="model",
    max_nodes_per_graph=0,
    random_seed=42,
):
    """
    Für jeden erklärten Target-Knoten:
    - explanation.node_mask_dict[current_type] ist eine [num_nodes_of_type, num_features]-Maske
    - daraus mitteln wir pro Node-Type über die Knoten dieses Typs
    - danach mitteln wir über alle erklärten Target-Knoten

    Ergebnis:
        {
            "H": {
                "avg_importance": [...],
                ...
            },
            "C": {
                "avg_importance": [...],
                ...
            }
        }
    """
    from tqdm.notebook import tqdm

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    base_model = load_trained_model(str(model_path), config, device)

    wrapped_model = NodeTypeRegressionWrapper(base_model, target_node_type)

    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )

    explainer = Explainer(
        model=wrapped_model,
        algorithm=GNNExplainer(epochs=int(epochs), lr=0.01, **globals().get('GNN_EXPLAINER_COEFFS', {'edge_size': 1e-4, 'edge_ent': 1e-4, 'node_feat_size': 1e-1, 'node_feat_ent': 1e-3})),
        explanation_type=expl_type,
        model_config=model_config,
        node_mask_type="attributes",
        edge_mask_type="object",
    )

    vectors_by_type = {}
    node_counts_by_type = {}

    from collections import defaultdict
    vectors_by_type = defaultdict(list)

    total_explanations = 0
    progress = tqdm(total=len(graph_indices), desc=f"GNNExplainer batch (target={target_node_type})")

    for graph_idx in graph_indices:
        validate_indices(len(dataset), graph_idx, "graph_idx")
        data = dataset[graph_idx].to(device)

        x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)

        if target_node_type not in x_dict:
            progress.update(1)
            continue

        target = None
        target_tensor = y_dict.get(target_node_type)
        if target_tensor is not None and expl_type == "phenomenon":
            target = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)

        num_nodes = x_dict[target_node_type].size(0)
        nodes_to_process = list(range(num_nodes))

        if 0 < int(max_nodes_per_graph) < num_nodes:
            nodes_to_process = random.sample(nodes_to_process, int(max_nodes_per_graph))

        for node_idx in nodes_to_process:
            if not (0 <= node_idx < num_nodes):
                continue
            if target is not None and torch.isnan(target[node_idx]):
                continue

            try:
                explanation = explainer(
                    x_dict,
                    edge_index_dict,
                    edge_attr_dict=edge_attr_dict,
                    target=target,
                    index=node_idx,
                )

                for current_type, mask in explanation.node_mask_dict.items():
                    if mask is None or mask.dim() != 2:
                        continue

                    # GNNExplainer liefert pro Node-Type eine Feature-Maske pro Knoten.
                    # Wir mitteln hier pro explained target node einfach über alle Knoten
                    # dieses Node-Types im Graphen / Teilgraphen.
                    mask = mask.detach().abs().cpu()
                    mean_vec = mask.mean(dim=0).numpy()
                    vectors_by_type[current_type].append(mean_vec)

                total_explanations += 1

            except Exception as e:
                print(f"Skipped graph {graph_idx}, target_type {target_node_type}, node {node_idx}: {e}")
                continue

        progress.update(1)

    progress.close()
    print(f"Completed {total_explanations} explanations for target={target_node_type}")

    all_types = sorted(vectors_by_type.keys())
    results = {}

    for nt in all_types:
        avg_imp = _safe_mean(vectors_by_type.get(nt, []))
        results[nt] = {
            "explainer_type": "gnn",
            "target_node_type": target_node_type,
            "epochs": int(epochs),
            "explanation_type": expl_type,

            "avg_importance": avg_imp,
            "avg_abs_importance": avg_imp,   # Alias für Kompatibilität
            "node_count": len(vectors_by_type.get(nt, [])),
        }

    return results


# ---------------------------------------------------------------------------
# Multi-target batch wrapper
# ---------------------------------------------------------------------------
def run_gnn_batch_analysis_all_targets_simple_compute_only():
    selected_data = batch_data_widget.value if 'batch_data_widget' in globals() else exp_data_widget.value
    data_path_obj = DATA_DIR / selected_data
    config_path_obj = MODELS_DIR / "config.pkl"

    if config_path_obj.exists() and data_path_obj.exists():
        config = load_config(str(config_path_obj))
        norm_stats_path_obj = MODELS_DIR / "norm_stats.pkl"
        edge_stats_path_obj = MODELS_DIR / "edge_stats.pkl"
        if norm_stats_path_obj.exists() and edge_stats_path_obj.exists():
            norm_stats, edge_stats = load_stats(str(norm_stats_path_obj), str(edge_stats_path_obj))
            dataset = build_dataset(str(data_path_obj), config, norm_stats=norm_stats, edge_stats=edge_stats)
            total_graphs = len(dataset)
        else:
            total_graphs = 1000
    else:
        total_graphs = 1000

    custom_graphs = _parse_graph_indices_gnn_like_ig(
        getattr(batch_graph_indices_widget, "value", None),
        total_graphs
    )

    if custom_graphs is not None:
        graph_indices = custom_graphs
        preview = graph_indices[:10]
        suffix = "..." if len(graph_indices) > 10 else ""
        print(f"Verarbeite benutzerdefinierte Graphen ({len(graph_indices)}): {preview}{suffix}")
    else:
        sample_graphs = int(batch_sample_graphs_widget.value)
        if sample_graphs <= 0 or sample_graphs >= total_graphs:
            graph_indices = list(range(total_graphs))
            print(f"Verarbeite alle {total_graphs} Graphen")
        else:
            graph_indices = list(range(sample_graphs))
            print(f"Verarbeite erste {sample_graphs} von {total_graphs} Graphen")

    target_node_types = list(batch_node_types_widget.value)
    if not target_node_types:
        print("❌ Fehler: Wähle mindestens einen Node Type aus")
        return None

    model_path = str(MODELS_DIR / exp_model_widget.value)
    data_path = str(DATA_DIR / selected_data)
    config_path = str(MODELS_DIR / "config.pkl")
    norm_stats_path = str(MODELS_DIR / "norm_stats.pkl")
    edge_stats_path = str(MODELS_DIR / "edge_stats.pkl")

    required_files = [model_path, data_path, config_path, norm_stats_path, edge_stats_path]
    missing_files = [f for f in required_files if not os.path.exists(f)]
    if missing_files:
        print(f"❌ Fehlende Dateien: {missing_files}")
        return None

    epochs = int(batch_epochs_widget.value)
    expl_type = str(batch_type_widget.value)
    max_nodes_per_graph = int(batch_sample_nodes_widget.value)

    results_per_target = {}
    global_rankings = {}

    for tgt_type in target_node_types:
        print(f"\n=== RUN für Target-Typ: {tgt_type} ===")

        tgt_results = run_gnn_batch_analysis_simple(
            model_path=model_path,
            data_path=data_path,
            config_path=config_path,
            norm_stats_path=norm_stats_path,
            edge_stats_path=edge_stats_path,
            graph_indices=graph_indices,
            target_node_type=tgt_type,
            epochs=epochs,
            expl_type=expl_type,
            max_nodes_per_graph=max_nodes_per_graph,
        )

        results_per_target[tgt_type] = tgt_results

        out_dir_batch = Path(PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()) / 'results' / 'gnn_batch'
        out_dir_batch.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = out_dir_batch / f"gnn_global_batch_{tgt_type}_{ts}.pt"
        torch.save(tgt_results, save_path)
        print(f"💾 Gespeichert: {save_path}")

        global_name = f"gnn_results_target{tgt_type}_simple"
        globals()[global_name] = tgt_results
        print(f"✅ Gespeichert als global: {global_name}")

        _print_batch_summary_gnn_simple(
            tgt_results,
            graph_indices,
            title=f"GNN Batch Summary — Target={tgt_type}",
        )

        _print_joint_top_features_gnn_simple(
            tgt_results,
            top_k=None,
            normalize_per_type=False,
        )

        print("[INFO] Visualisierung in separatem Schritt (aus gespeicherten Dateien).")

        # Build + persist reusable global ranking for correlation
        rank_rows = _build_joint_ranking_rows_gnn_simple(tgt_results, top_k=None)
        global_rankings[tgt_type] = rank_rows
        try:
            import json
            out_dir = Path(PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()) / 'results' / 'rankings'
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / f'gnn_global_ranking_{tgt_type}.json'
            out_path.write_text(json.dumps(rank_rows, indent=2))
            print(f"[INFO] Gespeichert: {out_path} ({len(rank_rows)} Zeilen)")
        except Exception as e:
            print(f"[WARN] Konnte Ranking-Datei für {tgt_type} nicht speichern: {e}")

    # publish ranking in-memory for correlation and save combined file
    globals()['gnn_global_print_rankings'] = global_rankings
    try:
        import json
        out_dir = Path(PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()) / 'results' / 'rankings'
        out_dir.mkdir(parents=True, exist_ok=True)
        combined = (global_rankings.get('H') or []) + (global_rankings.get('C') or [])
        combined_path = out_dir / 'gnn_global_ranking_combined.json'
        combined_path.write_text(json.dumps(combined, indent=2))
        print(f"[INFO] Gespeichert: {combined_path} ({len(combined)} Zeilen)")
    except Exception as e:
        print(f"[WARN] Konnte combined Ranking-Datei nicht speichern: {e}")

    global gnn_batch_results_per_target, gnn_batch_saved_files
    gnn_batch_results_per_target = results_per_target
    out_dir_batch = Path(PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()) / 'results' / 'gnn_batch'
    gnn_batch_saved_files = {}
    for tgt in target_node_types:
        pattern = str(out_dir_batch / f"gnn_global_batch_{tgt}_*.pt")
        import glob
        files = glob.glob(pattern)
        gnn_batch_saved_files[tgt] = (max(files, key=lambda x: Path(x).stat().st_mtime) if files else None)
    print("\n✅ Alle GNN-Target-Runs abgeschlossen und gespeichert.")
    return results_per_target

# ---------------------------------------------------------------------------
# Fallback Widget-Initialisierung (falls nur exp_* Widgets geladen sind)
# ---------------------------------------------------------------------------
if 'batch_data_widget' not in globals():
    if 'exp_data_widget' in globals():
        batch_data_widget = widgets.Dropdown(
            options=exp_data_widget.options,
            value=exp_data_widget.value,
            description='Data:',
            layout=widgets.Layout(width='300px'),
        )
    else:
        _data_options = sorted([f.name for f in DATA_DIR.glob('*.pt')]) if 'DATA_DIR' in globals() else []
        batch_data_widget = widgets.Dropdown(
            options=_data_options,
            value=_data_options[0] if _data_options else None,
            description='Data:',
            layout=widgets.Layout(width='300px'),
        )

if 'batch_type_widget' not in globals():
    _exp_type = exp_type_widget.value if 'exp_type_widget' in globals() else 'model'
    batch_type_widget = widgets.Dropdown(
        options=['model', 'phenomenon'],
        value=_exp_type if _exp_type in ['model', 'phenomenon'] else 'model',
        description='Type:',
        layout=widgets.Layout(width='220px'),
    )

if 'batch_node_types_widget' not in globals():
    batch_node_types_widget = widgets.SelectMultiple(
        options=['H', 'C', 'Others'],
        value=('H', 'C'),
        description='Node Types:',
        layout=widgets.Layout(width='260px', height='90px'),
    )

if 'batch_epochs_widget' not in globals():
    _exp_epochs = int(exp_epochs_widget.value) if 'exp_epochs_widget' in globals() else 30
    batch_epochs_widget = widgets.IntSlider(
        value=max(1, _exp_epochs),
        min=1,
        max=200,
        step=1,
        description='Epochs:',
        continuous_update=False,
        layout=widgets.Layout(width='260px'),
    )

if 'batch_sample_graphs_widget' not in globals():
    batch_sample_graphs_widget = widgets.IntSlider(
        value=100,
        min=1,
        max=2000,
        step=1,
        description='Graphs:',
        continuous_update=False,
        layout=widgets.Layout(width='260px'),
    )

if 'batch_sample_nodes_widget' not in globals():
    batch_sample_nodes_widget = widgets.IntSlider(
        value=100,
        min=1,
        max=5000,
        step=1,
        description='Nodes/Graph:',
        continuous_update=False,
        layout=widgets.Layout(width='260px'),
    )

if 'batch_graph_indices_widget' not in globals():
    batch_graph_indices_widget = widgets.Text(
        value='',
        description='Graph IDs:',
        placeholder='z.B. 0,1,5-12',
        layout=widgets.Layout(width='360px'),
    )

# ---------------------------------------------------------------------------
# GNN Global Batch Widgets (Style wie IG)
# ---------------------------------------------------------------------------
gnn_global_controls = widgets.VBox([
    widgets.HTML('<h4>GNN Global Batch Einstellungen</h4>'),
    widgets.HBox([exp_model_widget, batch_data_widget, batch_type_widget]),
    widgets.HBox([batch_node_types_widget, batch_epochs_widget, batch_sample_graphs_widget]),
    widgets.HBox([batch_sample_nodes_widget, batch_graph_indices_widget]),
], layout=widgets.Layout(width='100%'))

display(gnn_global_controls)


# ---------------------------------------------------------------------------
# Schritt 1: Berechnen + speichern
# ---------------------------------------------------------------------------
gnn_batch_compute_button = widgets.Button(
    description="GNN Batch: berechnen + speichern",
    button_style="primary",
    icon="save",
)

gnn_batch_compute_output = widgets.Output()

def on_gnn_batch_compute_click(b):
    with gnn_batch_compute_output:
        clear_output()
        run_gnn_batch_analysis_all_targets_simple_compute_only()

gnn_batch_compute_button.on_click(on_gnn_batch_compute_click)

display(gnn_batch_compute_button)
display(gnn_batch_compute_output)


Button(button_style='primary', description='GNN Batch: berechnen + speichern', icon='save', style=ButtonStyle(…

Output()

In [ ]:
from pathlib import Path
import glob
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------------------------
# Farben / Stil
# ---------------------------------------------------------------------------
UNI_BLUE    = "#004e9f"
UNI_BLUE_50 = "#7fa6cf"
UNI_BLUE_25 = "#bfd3e7"
UNI_YELLOW  = "#fcba00"
UNI_GREY    = "#909085"
UNI_GREY_25 = "#e3e3e0"
UNI_BLACK   = "#1a1a1a"

TYPE_COLORS = {
    "H": UNI_BLUE,
    "C": UNI_YELLOW,
    "Others": UNI_GREY,
}

def _apply_uni_style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor("#fafaf8")
    ax.figure.patch.set_facecolor("white")
    ax.grid(color=UNI_GREY_25, linewidth=0.8, linestyle="-", alpha=0.9)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=UNI_BLACK, labelsize=9)
    if title:
        ax.set_title(title, color=UNI_BLUE, fontsize=12, fontweight="bold", pad=10, loc="left")
    if xlabel:
        ax.set_xlabel(xlabel, color=UNI_BLACK, fontsize=10, labelpad=6)
    if ylabel:
        ax.set_ylabel(ylabel, color=UNI_BLACK, fontsize=10, labelpad=6)

def _add_total_labels_right(ax, y_pos, total_values, max_abs, fontsize=7.5):
    offset = max_abs * 0.01 if max_abs > 0 else 0.01
    for y, total in zip(y_pos, total_values):
        if abs(total) <= max_abs * 0.01:
            continue
        ax.text(
            total + offset,
            y,
            f"{total:.3f}",
            ha="left",
            va="center",
            fontsize=fontsize,
            color=UNI_BLACK,
        )



# ---------------------------------------------------------------------------
# Visualisierung: pro Target-Typ einfach ein Diagramm je Node-Type
# ---------------------------------------------------------------------------
def plot_gnn_feature_masks_by_type(batch_results, target_node_type=None, show_all=True, top_k=25):
    import matplotlib.pyplot as plt
    import numpy as np

    available_types = [nt for nt, res in batch_results.items() if len(res.get("avg_importance", [])) > 0]

    if not available_types:
        print("⚠️ Keine avg_importance-Daten gefunden.")
        return

    desired_order = [t for t in ["H", "C", "Others"] if t in available_types]
    desired_order += [t for t in available_types if t not in desired_order]

    for node_type in desired_order:
        vals = np.array(batch_results[node_type].get("avg_importance", []), dtype=float)
        if vals.size == 0:
            continue

        order = np.argsort(vals)[::-1]
        if not show_all and top_k is not None and top_k > 0:
            order = order[:top_k]

        names = feature_names_for(node_type, len(vals))
        labels = [names[i] for i in order]
        plot_vals = vals[order]

        fig_h = max(8, len(order) * 0.35)
        fig, ax = plt.subplots(figsize=(16, fig_h))
        fig.subplots_adjust(left=0.22, right=0.96, top=0.94, bottom=0.06)

        ax.barh(
            np.arange(len(order)),
            plot_vals,
            color=TYPE_COLORS.get(node_type, UNI_GREY),
            alpha=0.85,
            height=0.65,
            edgecolor="white",
            linewidth=0.4,
        )

        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()

        title = f"GNNExplainer feature importance | Target={target_node_type} | NodeType={node_type}"
        _apply_uni_style(
            ax,
            title=title,
            xlabel="Average feature mask importance",
            ylabel="Features",
        )

        max_val = plot_vals.max() if len(plot_vals) and plot_vals.max() != 0 else 1.0
        _add_total_labels_right(ax, np.arange(len(order)), plot_vals, max_val)
        ax.set_xlim(0, max_val * 1.12)

        plt.tight_layout()
        plt.show()




def _latest_gnn_batch_file(output_dir: str, target_node_type: str):
    pattern = str(Path(output_dir) / f"gnn_global_batch_{target_node_type}_*.pt")
    files = glob.glob(pattern)
    if not files:
        return None
    return max(files, key=lambda p: Path(p).stat().st_mtime)


def run_gnn_batch_analysis_all_targets_simple_plot_only_from_saved():
    global gnn_batch_results_per_target, gnn_batch_saved_files
    target_node_types = list(batch_node_types_widget.value)
    if not target_node_types:
        print("❌ Fehler: Wähle mindestens einen Node Type aus")
        return None

    output_dir = str((Path(PROJECT_ROOT) if 'PROJECT_ROOT' in globals() else Path.cwd()) / 'results' / 'gnn_batch')
    saved_files = {}
    if 'gnn_batch_saved_files' in globals() and isinstance(gnn_batch_saved_files, dict):
        saved_files.update(gnn_batch_saved_files)

    loaded_results = {}
    for tgt_type in target_node_types:
        fpath = saved_files.get(tgt_type) or _latest_gnn_batch_file(output_dir, tgt_type)
        if not fpath:
            print(f"⚠️ Keine gespeicherte Datei für Target={tgt_type} gefunden.")
            continue
        try:
            payload = torch.load(fpath, weights_only=False)
            if not isinstance(payload, dict):
                print(f"⚠️ Ungültige Datei für Target={tgt_type}: {fpath}")
                continue
            loaded_results[tgt_type] = payload
            saved_files[tgt_type] = fpath
            print(f"📂 Geladen ({tgt_type}): {fpath}")
        except Exception as e:
            print(f"⚠️ Laden fehlgeschlagen für {tgt_type}: {e}")

    if not loaded_results:
        print("❌ Keine Ergebnisse zum Plotten geladen.")
        return None

    for tgt_type, tgt_results in loaded_results.items():
        print(f"=== PLOT für Target-Typ: {tgt_type} (aus Datei) ===")
        _print_batch_summary_gnn_simple(
            tgt_results,
            [],
            title=f"GNN Batch Summary — Target={tgt_type} — aus gespeicherter Datei",
        )
        _print_joint_top_features_gnn_simple(
            tgt_results,
            top_k=None,
            normalize_per_type=False,
        )
        print("Erstelle Visualisierung (aus gespeicherten Erklärungen)...")
        try:
            plot_gnn_feature_masks_by_type(
                tgt_results,
                target_node_type=tgt_type,
                show_all=True,
                top_k=25,
            )
            print("✅ Visualisierung erfolgreich erstellt!")
        except Exception as e:
            print(f"⚠️ Fehler bei Visualisierung: {e}")

    gnn_batch_results_per_target = loaded_results
    gnn_batch_saved_files = saved_files
    return loaded_results


# ---------------------------------------------------------------------------
# Schritt 2: Aus gespeicherten Dateien visualisieren
# ---------------------------------------------------------------------------
gnn_batch_plot_button = widgets.Button(
    description="GNN Batch: aus Datei plotten",
    button_style="primary",
    icon="line-chart",
)

gnn_batch_plot_output = widgets.Output()

def on_gnn_batch_plot_click(b):
    with gnn_batch_plot_output:
        clear_output()
        run_gnn_batch_analysis_all_targets_simple_plot_only_from_saved()

gnn_batch_plot_button.on_click(on_gnn_batch_plot_click)

display(gnn_batch_plot_button)
display(gnn_batch_plot_output)

Button(button_style='primary', description='GNN Batch: aus Datei plotten', icon='line-chart', style=ButtonStyl…

Output()

In [ ]:
def load_sdf_atoms(path):
 
    atoms = []
    with open(path) as f:
        lines = f.readlines()

    if len(lines) < 5:
        return atoms

    # Counts-Zeile (typischerweise Zeile 4, Index 3)
    counts_line = lines[3]
    try:
        n_atoms = int(counts_line[0:3])
    except ValueError:
        # Falls das Format anders ist -> nichts zurückgeben
        return atoms

    # Atomzeilen beginnen ab Zeile 5 (Index 4)
    atom_lines = lines[4 : 4 + n_atoms]

    for idx, line in enumerate(atom_lines):
        # Standard SDF-V2000: x,y,z in Spalten 0:10,10:20,20:30
        try:
            x = float(line[0:10])
            y = float(line[10:20])
            z = float(line[20:30])
            symbol = line[31:34].strip()
        except ValueError:
            continue
        atoms.append(
            {
                "index": idx,
                "symbol": symbol,
                "x": x,
                "y": y,
                "z": z,
            }
        )
    return atoms


# Pure blue→red gradient without green for this cell
# v=0 → blue, v=1 → red
def importance_to_color(v):
    v = max(0.0, min(1.0, float(v)))
    r = int(v * 255)
    g = 0
    b = int((1.0 - v) * 255)
    return f'rgb({r},{g},{b})'


if "explanation_summary" in globals() and "explanation_obj" in globals():
    import py3Dmol
    import pickle
    from pathlib import Path

    PROJECT_ROOT = Path.cwd().parent
    DATA_DIR = PROJECT_ROOT / "data"
    MODELS_DIR = PROJECT_ROOT / "models"

    graph_idx = explanation_summary["graph_idx"]
    node_type = explanation_summary["node_type"]
    node_idx = explanation_summary["node_idx"]

    # Dataset laden
    data_file = exp_data_widget.value
    config_path = MODELS_DIR / "config.pkl"
    norm_stats_path = MODELS_DIR / "norm_stats.pkl"
    edge_stats_path = MODELS_DIR / "edge_stats.pkl"

    with open(config_path, "rb") as f:
        config = pickle.load(f)
    with open(norm_stats_path, "rb") as f:
        norm_stats = pickle.load(f)
    with open(edge_stats_path, "rb") as f:
        edge_stats = pickle.load(f)

    dataset = build_dataset(str(DATA_DIR / data_file), config, norm_stats=norm_stats, edge_stats=edge_stats)
    nx_g = dataset.nx_graphs[graph_idx]

    compound = nx_g.graph.get("compound", 1)
    structure = nx_g.graph.get("structure", 1)
    subfolder = f"{int(compound):03d}"

    sdf_name = f"{subfolder}_{int(structure):02d}.sdf"
    sdf_path = DATA_DIR / "orca_xyz_formate" / subfolder / sdf_name
    print(f"Looking for SDF file: {sdf_path}")

    if sdf_path.exists():
        atoms = load_sdf_atoms(str(sdf_path))

        with open(str(sdf_path)) as f:
            sdf_block = f.read()

        # ERSTELLE ATOM_INDEX_DICT LOKAL aus dem NetworkX-Graphen
        atom_index_dict = {"H": [], "C": [], "Others": []}
        for node in nx_g.nodes():
            element = nx_g.nodes[node]["element"]
            if element in ["H", "C"]:
                atom_index_dict[element].append(node)
            else:
                atom_index_dict["Others"].append(node)
        
        # Konvertiere zu Tensoren
        for ntype in atom_index_dict:
            if atom_index_dict[ntype]:
                atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
            else:
                atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

        # Fokussiertes Atom
        if node_type in atom_index_dict and atom_index_dict[node_type].numel() > node_idx:
            focus_index = int(atom_index_dict[node_type][node_idx].item())
        else:
            print(f"Fehler: node_idx {node_idx} out of bounds für {node_type}")
            focus_index = None

        # ERSTELLE EDGE_INDEX_DICT LOKAL aus dem HeteroData Objekt
        # Dafür brauchen wir das ursprüngliche data Objekt
        data = dataset[graph_idx]  # HeteroData Objekt
        edge_index_dict = {}
        for store in data.edge_stores:
            key = store._key
            edge_index_dict[key] = store.edge_index

        # Edge-Importances pro Atom aggregieren (Maximum)
        edge_masks = explanation_obj.edge_mask_dict if hasattr(explanation_obj, 'edge_mask_dict') else {}
        atom_importance = {}

        # Fuer jeden Node-Type
        for ntype, atom_indices in atom_index_dict.items():
            if atom_indices.numel() == 0:
                continue
            
            # Edge-Types, die diesen Node-Type betreffen
            relevant_edge_types = []
            for edge_type in edge_index_dict.keys():
                src_type, _, dst_type = edge_type
                if src_type == ntype or dst_type == ntype:
                    relevant_edge_types.append(edge_type)
            
            # Fuer jeden Node des Types
            for local_idx in range(len(atom_indices)):
                sdf_idx = int(atom_indices[local_idx].item())
                
                max_importance = 0.0
                
                # Alle relevanten Edge-Types durchgehen
                for edge_type in relevant_edge_types:
                    if edge_type not in edge_masks:
                        continue
                        
                    ei = edge_index_dict[edge_type]
                    mask = edge_masks[edge_type]
                    
                    # Kanten finden, die diesen Node betreffen
                    src_type, _, dst_type = edge_type
                    
                    if src_type == ntype:
                        # Outgoing edges von diesem Node
                        for edge_idx in range(ei.size(1)):
                            if ei[0, edge_idx] == local_idx:
                                max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                                
                    if dst_type == ntype:
                        # Incoming edges zu diesem Node
                        for edge_idx in range(ei.size(1)):
                            if ei[1, edge_idx] == local_idx:
                                max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                
                # Maximaler Importance-Wert der anliegenden Kanten
                atom_importance[sdf_idx] = max_importance

        print(f"Colored atoms (count): {len(atom_importance)}")

        # py3Dmol Viewer Setup
        view = py3Dmol.view(width=600, height=500)
        view.addModel(sdf_block, "sdf")
        view.setStyle({
            "stick": {
                "color": "lightgray",
                "radius": 0.2,
                "doubleBondScaling": 0.4,
                "singleBonds": False,
            }
        })

        # Labels für H + Heteroatome
        if atoms:
            for atom in atoms:
                if atom["symbol"] != "C":
                    view.addLabel(
                        atom["symbol"],
                        {
                            "position": {
                                "x": atom["x"],
                                "y": atom["y"],
                                "z": atom["z"],
                            },
                            "fontSize": 12,
                            "fontColor": "black",
                            "backgroundColor": "white",
                            "showBackground": True,
                        },
                    )

        # Atome einfärben nach Importance
        if atom_importance:
            vals = list(atom_importance.values())
            norm_atoms = make_normalizer(vals)
            for atom_idx, imp in atom_importance.items():
                v = norm_atoms(imp)
                color = importance_to_color(v)
                view.addStyle(
                    {"index": atom_idx},
                    {"stick": {"color": color, "radius": 0.25}},
                )
                view.addStyle(
                    {"index": atom_idx},
                    {"sphere": {"color": color, "radius": 0.4}},
                )

        # Wichtige Kanten zeichnen
        important_edges = []
        if "important_edges" in explanation_summary:
            for e in explanation_summary["important_edges"]:
                important_edges.append({
                    "edge_type": e["edge_type"],
                    "importance": e["importance"],
                    "edge_position": e.get("edge_position", 0),
                })

        if important_edges and atoms:
            imps = [abs(e["importance"]) for e in important_edges]
            norm_edges = make_normalizer(imps)
            
            for e in important_edges:
                edge_type = tuple(e["edge_type"])
                pos = int(e["edge_position"])
                imp = float(e["importance"])
                
                if abs(imp) <= 0.01:
                    continue
                    
                if edge_type not in edge_index_dict:
                    continue

                ei = edge_index_dict[edge_type]
                if pos < 0 or pos >= ei.size(1):
                    continue

                u_local = int(ei[0, pos].item())
                v_local = int(ei[1, pos].item())

                src_type, _, dst_type = edge_type

                # Mapping lokal->SDF ueber das lokale atom_index_dict
                if src_type not in atom_index_dict or atom_index_dict[src_type].numel() <= u_local:
                    continue
                if dst_type not in atom_index_dict or atom_index_dict[dst_type].numel() <= v_local:
                    continue

                u_sdf = int(atom_index_dict[src_type][u_local].item())
                v_sdf = int(atom_index_dict[dst_type][v_local].item())

                if not (0 <= u_sdf < len(atoms) and 0 <= v_sdf < len(atoms)):
                    continue

                v = norm_edges(abs(imp))
                color = importance_to_color(v)

                ai = atoms[u_sdf]
                aj = atoms[v_sdf]
                mid = {
                    "x": (ai["x"] + aj["x"]) / 2,
                    "y": (ai["y"] + aj["y"]) / 2,
                    "z": (ai["z"] + aj["z"]) / 2,
                }

                view.addLine({
                    "start": {"x": ai["x"], "y": ai["y"], "z": ai["z"]},
                    "end": mid,
                    "color": color,
                    "linewidth": 8,
                })

        # Fokus-Atom hervorheben
        if focus_index is not None and 0 <= focus_index < len(atoms):
            fa = atoms[focus_index]
            view.addStyle(
                {"index": focus_index},
                {"sphere": {"color": "yellow", "radius": 0.6}},
            )
            view.addLabel(
                str(focus_index),
                {
                    "fontSize": 14,
                    "fontColor": "black",
                    "backgroundColor": "white",
                    "showBackground": True,
                    "position": {
                        "x": fa["x"],
                        "y": fa["y"],
                        "z": fa["z"],
                    },
                },
            )

        view.zoomTo()
        view.show()
    else:
        print(f"SDF-Datei für Graph {graph_idx} nicht gefunden: {sdf_path}")
else:
    print("Keine Erklärung vorhanden oder Visualisierung mit Beispiel.")

Keine Erklärung vorhanden oder Visualisierung mit Beispiel.


---
## 4. Hilfsfunktionen

In [ ]:
def make_normalizer(values, default=1.0):
    if not values:
        return lambda v: default
    vmin = min(values)
    vmax = max(values)

    def norm(v):
        if vmax == vmin:
            return default
        return (v - vmin) / (vmax - vmin)

    return norm


def importance_to_color(v):
    """
    Mappt einen Wert in [0,1] auf ein rot-blau-Farbspektrum:
      v=0 -> blau, v=1 -> rot.
    """
    v = max(0.0, min(1.0, v))
    r = int(v * 255)
    b = int((1 - v) * 255)
    g = 0
    return f"rgb({r},{g},{b})"



def show_dataset_info(data_file):
    """Zeigt Informationen über einen Datensatz an."""
    data_path = DATA_DIR / data_file
    
    if not data_path.exists():
        print(f"Datei nicht gefunden: {data_path}")
        return
    
    # Config laden falls vorhanden
    config_path = MODELS_DIR / 'config.pkl'
    if config_path.exists():
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        if not isinstance(config, dict):
            config = vars(config) if hasattr(config, '__dict__') else dict(config)
        normalize_nodes = config.get('normalize_node_features', False)
        normalize_edges = config.get('normalize_edge_features', False)
    else:
        normalize_nodes = False
        normalize_edges = False
    
    # Dataset laden
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=data_file,
        normalize_node_features=normalize_nodes,
        normalize_edge_features=normalize_edges
    )
    
    print(f"Datensatz: {data_file}")
    print(f"Anzahl Graphen: {len(dataset)}")
    
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"\nBeispiel Graph (Index 0):")
        print(f"  Node Types: {sample.node_types}")
        for ntype in sample.node_types:
            print(f"    {ntype}: {sample[ntype].x.shape[0]} Knoten, {sample[ntype].x.shape[1]} Features")
        print(f"  Edge Types: {len(sample.edge_types)}")


# Widget für Dataset Info
info_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Datensatz:',
    style={'description_width': 'initial'}
)

info_button = widgets.Button(
    description='Info anzeigen',
    button_style='',
    icon='info'
)

info_output = widgets.Output()

def on_info_click(b):
    with info_output:
        clear_output()
        show_dataset_info(info_data_widget.value)

info_button.on_click(on_info_click)

display(widgets.VBox([
    widgets.HTML('<h4>Datensatz-Informationen</h4>'),
    info_data_widget,
    info_button,
    info_output
]))

## Integrated Gradients Explainer

In [ ]:
# IG Explainer Widgets
ig_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

ig_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

ig_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

ig_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

ig_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

ig_n_steps_widget = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description='Integration Steps:',
    style={'description_width': 'initial'}
)

ig_baseline_widget = widgets.Dropdown(
    options=['zero', 'mean', 'random', 'min', 'max', 'scientific'],
    value='scientific',
    description='Baseline Type:',
    style={'description_width': 'initial'}
)

ig_train_split_widget = widgets.Text(
    value='models/graph_split.pkl',
    description='Train Split (.pkl):',
    placeholder='optional, z.B. models/graph_split.pkl',
    style={'description_width': 'initial'}
)

ig_include_neighbors_widget = widgets.Checkbox(
    value=True,
    description='Include Neighbors (3-hop)',
    style={'description_width': 'initial'}
)

ig_k_hops_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    description='Number of Hops:',
    style={'description_width': 'initial'}
)

ig_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout
ig_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    ig_model_widget,
    ig_data_widget,
    ig_graph_idx_widget,
    ig_node_type_widget,
    ig_node_idx_widget
])

ig_right = widgets.VBox([
    widgets.HTML('<h4>IG Einstellungen</h4>'),
    ig_n_steps_widget,
    ig_baseline_widget,
    ig_train_split_widget,
    ig_include_neighbors_widget,
    ig_k_hops_widget,
    ig_output_dir_widget
])

display(widgets.HBox([ig_left, ig_right]))


In [ ]:
from scripts.explainer.ig_explainer import compute_ig_explanation
from scripts.explainer.baselines import load_or_compute_medians, load_or_compute_element_distribution
from feature_visualization import get_feature_names, plot_single_feature_importance
import matplotlib.pyplot as plt

def run_ig_explanation():
    """Generiert eine IG Erklärung für den ausgewählten Knoten und optional seine Nachbarn."""

    device = get_device(None)

    # Pfade
    model_path = MODELS_DIR / ig_model_widget.value
    data_path = DATA_DIR / ig_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'

    # Config und Stats laden
    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))

    # Dataset erstellen
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    validate_indices(len(dataset), ig_graph_idx_widget.value, "graph_idx")

    baseline_type = ig_baseline_widget.value
    median_global = None
    median_by_element = None
    train_graph_indices = None
    train_split_path = None

    if baseline_type == 'scientific':
        train_split_raw = (ig_train_split_widget.value or '').strip()

        if train_split_raw:
            train_split_path = Path(train_split_raw)
            if not train_split_path.is_absolute():
                train_split_path = PROJECT_ROOT / train_split_path
            if train_split_path.exists():
                with open(train_split_path, 'rb') as handle:
                    split_data = pickle.load(handle)
                    # Extract train indices from dict if needed
                    if isinstance(split_data, dict):
                        train_graph_indices = split_data.get('train_graph_indices', [])
                    else:
                        train_graph_indices = split_data
                print(f"Scientific baseline: nutze Train-Split aus {train_split_path} ({len(train_graph_indices)} Graphen)")
            else:
                print(f"⚠️ Train-Split-Datei nicht gefunden: {train_split_path}")

        if train_graph_indices is None:
            train_graph_indices = list(range(int(0.8 * len(dataset))))
            print(f"⚠️ Scientific baseline ohne Train-Split-Datei: nutze 80%-Fallback ({len(train_graph_indices)} Graphen)")

        median_global, median_by_element = load_or_compute_medians(
            dataset=dataset,
            train_graph_indices=train_graph_indices,
            node_type=ig_node_type_widget.value,
            cache_dir=str(PROJECT_ROOT / "baselines"),
        )
        elem_distribution = load_or_compute_element_distribution(
            dataset=dataset,
            train_graph_indices=train_graph_indices,
            cache_dir=str(PROJECT_ROOT / "baselines"),
        )
    else:
        elem_distribution = None

    # Daten laden
    data = dataset[ig_graph_idx_widget.value].to(device)
    x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)

    # Modell laden
    base_model = load_trained_model(str(model_path), config, device)

    # IG-Erklärung berechnen (mit oder ohne Nachbarn)
    explanation_result = compute_ig_explanation(
        base_model=base_model,
        data=data,
        node_type=ig_node_type_widget.value,
        node_idx=ig_node_idx_widget.value,
        x_dict=x_dict,
        edge_index_dict=edge_index_dict,
        edge_attr_dict=edge_attr_dict,
        device=device,
        n_steps=ig_n_steps_widget.value,
        baseline_type=baseline_type,
        include_neighbors=ig_include_neighbors_widget.value,
        k_hops=ig_k_hops_widget.value,
        median_global=median_global,
        median_by_element=median_by_element,
        elem_distribution=elem_distribution,
        dataset=dataset,
        train_graph_indices=train_graph_indices,
        train_split_path=str(train_split_path) if train_split_path else None,
    )

    # Vorhersage holen
    with torch.no_grad():
        predictions = base_model(x_dict, edge_index_dict, edge_attr_dict)
        node_prediction = float(predictions[ig_node_type_widget.value][ig_node_idx_widget.value].item())

    # Ergebnis formatieren
    summary = {
        "graph_idx": ig_graph_idx_widget.value,
        "node_type": ig_node_type_widget.value,
        "node_idx": ig_node_idx_widget.value,
        "prediction": node_prediction,
        "n_steps": ig_n_steps_widget.value,
        "baseline_type": baseline_type,
        "include_neighbors": ig_include_neighbors_widget.value,
        "k_hops": ig_k_hops_widget.value,
        "feature_importance": explanation_result["selected_node"]["attributions"],
        # Captum completeness diagnostic (path integral vs. prediction diff)
        "convergence_delta": explanation_result.get("convergence_delta"),
        # Alias for readability in the printout
        "completeness": explanation_result.get("convergence_delta"),
    }

    # Speichern
    output_dir = PROJECT_ROOT / ig_output_dir_widget.value
    ensure_dir(str(output_dir))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = (
        f"ig_explainer_{ig_node_type_widget.value}"
        f"_n{ig_node_idx_widget.value}"
        f"_g{ig_graph_idx_widget.value}"
        f"_{timestamp}"
    )

    torch.save(explanation_result, output_dir / f"{base_name}.pt")
    with open(output_dir / f"{base_name}.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    print(f"\n✅ IG Erklärung gespeichert in: {output_dir}")
    return summary, explanation_result

# IG Explain Button
ig_explain_button = widgets.Button(
    description='IG Erklärung generieren',
    button_style='info',
    icon='search-plus'
)

ig_explain_output = widgets.Output()

def on_ig_explain_click(_b):
    with ig_explain_output:
        clear_output()
        try:
            global ig_explanation_summary, ig_explanation_obj
            ig_explanation_summary, ig_explanation_obj = run_ig_explanation()
            print("\n" + "=" * 50)
            print("IG ERKLÄRUNGSZUSAMMENFASSUNG")
            print("=" * 50)
            print(f"Graph Index: {ig_explanation_summary['graph_idx']}")
            print(f"Node Type: {ig_explanation_summary['node_type']}")
            print(f"Node Index: {ig_explanation_summary['node_idx']}")
            print(f"Vorhersage: {ig_explanation_summary['prediction']:.4f}")
            print(f"Integration Steps: {ig_explanation_summary['n_steps']}")
            print(f"Baseline Type: {ig_explanation_summary['baseline_type']}")
            # Captum completeness diagnostics
            if ig_explanation_summary.get('convergence_delta') is not None:
                print(f"Convergence Δ: {ig_explanation_summary['convergence_delta']:.6e}")
                print(f"Completeness (alias): {ig_explanation_summary['completeness']:.6e}")
            
            # Selected node features
            print("\n" + "=" * 50)
            print("SELECTED NODE FEATURE IMPORTANCE")
            print("=" * 50)
            feature_imp = ig_explanation_summary['feature_importance']
            sorted_features = sorted(enumerate(feature_imp), key=lambda x: abs(x[1]), reverse=True)
            feature_names = get_feature_names(ig_explanation_summary['node_type'])
            
            print(f"\nTop 10 wichtigste Features für ausgewählten Node:")
            for i, (idx, imp) in enumerate(sorted_features[:10]):
                name = feature_names[idx] if idx < len(feature_names) else f'Feature_{idx}'
                print(f"  {name}: {imp:.6f}")
            
            # Balkendiagramm für ausgewählten Node
            try:
                plot_single_feature_importance(
                    np.array(feature_imp), 
                    ig_explanation_summary['node_type'], 
                    f" (Graph {ig_explanation_summary['graph_idx']}, Node {ig_explanation_summary['node_idx']} - SELECTED)"
                )
            except Exception as e:
                print(f"Fehler bei Diagramm-Erstellung: {e}")
            
            # Neighbor information
            if ig_explanation_summary.get('include_neighbors') and 'neighbors' in ig_explanation_obj:
                neighbors = ig_explanation_obj['neighbors']
                print("\n" + "=" * 50)
                print(f"NACHBAR-EINFLUSS AUF AUSGEWÄHLTEN KNOTEN")
                print(f"({len(neighbors)} neighbors within {ig_explanation_summary['k_hops']} hops)")
                print("=" * 50)
                print(f"\nZeigt: Wie die Features der Nachbarn die Vorhersage von {ig_explanation_summary['node_type']}[{ig_explanation_summary['node_idx']}] beeinflussen")
                
                # Group neighbors by type
                neighbors_by_type = {}
                for neighbor in neighbors:
                    ntype = neighbor['node_type']
                    if ntype not in neighbors_by_type:
                        neighbors_by_type[ntype] = []
                    neighbors_by_type[ntype].append(neighbor)
                
                print(f"\nNachbar-Verteilung:")
                for ntype, nlist in neighbors_by_type.items():
                    print(f"  {ntype}: {len(nlist)} neighbors")
                
                # Create visualizations for neighbors
                print("\nGeneriere Feature Importance Plots für Nachbar-Einfluss...")
                print(f"(Alle Plots zeigen Einfluss auf {ig_explanation_summary['node_type']}[{ig_explanation_summary['node_idx']}])")
                
                # Plot a few example neighbors from each type
                for ntype, nlist in neighbors_by_type.items():
                    # Show up to 3 neighbors of each type
                    for i, neighbor in enumerate(nlist[:3]):
                        neighbor_feat_names = get_feature_names(neighbor['node_type'])
                        neighbor_attrs = np.array(neighbor['attributions'])
                        
                        try:
                            plot_single_feature_importance(
                                neighbor_attrs,
                                neighbor['node_type'],
                                f" - Einfluss von {neighbor['node_type']}[{neighbor['node_idx']}] auf Selected Node"
                            )
                        except Exception as e:
                            print(f"  Error plotting neighbor {ntype}[{neighbor['node_idx']}]: {e}")
                
                # Summary statistics - using robust metrics
                print("\n" + "=" * 50)
                print(f"AGGREGIERTE NACHBAR-STATISTIKEN")
                print(f"(Einfluss auf {ig_explanation_summary['node_type']}[{ig_explanation_summary['node_idx']}])")
                print("=" * 50)
                
                if 'neighbor_stats' in ig_explanation_obj:
                    neighbor_stats = ig_explanation_obj['neighbor_stats']
                    
                    for ntype, stats in neighbor_stats.items():
                        print(f"\n{ntype} neighbors (n={stats['num_neighbors']}):")
                        feat_names = get_feature_names(ntype)
                        
                        # Use mean absolute attribution (robust metric)
                        mean_abs_imp = np.array(stats['mean_abs_importance'])
                        std_abs_imp = np.array(stats['std_abs_importance'])
                        sign_stab = np.array(stats['sign_stability'])
                        pos_count = np.array(stats['positive_count'])
                        neg_count = np.array(stats['negative_count'])
                        
                        # Sort by mean absolute importance
                        sorted_indices = sorted(range(len(mean_abs_imp)), key=lambda x: mean_abs_imp[x], reverse=True)
                        
                        print(f"  Top 5 einflussreichste {ntype}-Features (Mean |Attribution|):")
                        for i, idx in enumerate(sorted_indices[:5]):
                            name = feat_names[idx] if idx < len(feat_names) else f'Feature_{idx}'
                            mean_abs = mean_abs_imp[idx]
                            std_abs = std_abs_imp[idx]
                            sign_pct = sign_stab[idx] * 100
                            pos = int(pos_count[idx])
                            neg = int(neg_count[idx])
                            
                            print(f"    {name}:")
                            print(f"      Mean |IG|: {mean_abs:.6f} ± {std_abs:.6f}")
                            print(f"      Sign-Stabilität: {sign_pct:.1f}% ({pos} pos / {neg} neg)")
                else:
                    # Fallback to simple calculation if stats not available
                    for ntype, nlist in neighbors_by_type.items():
                        all_attrs = np.array([n['attributions'] for n in nlist])
                        mean_abs_importance = np.mean(np.abs(all_attrs), axis=0)
                        
                        print(f"\n{ntype} neighbors (n={len(nlist)}):")
                        sorted_indices = sorted(range(len(mean_abs_importance)), key=lambda x: mean_abs_importance[x], reverse=True)
                        feat_names = get_feature_names(ntype)
                        
                        print(f"  Top 5 einflussreichste {ntype}-Features (Mean |Attribution|):")
                        for i, idx in enumerate(sorted_indices[:5]):
                            name = feat_names[idx] if idx < len(feat_names) else f'Feature_{idx}'
                            print(f"    {name}: {mean_abs_importance[idx]:.6f}")

        except Exception as e:
            print(f"\n❌ Fehler bei IG Erklärung: {e}")
            import traceback
            traceback.print_exc()

ig_explain_button.on_click(on_ig_explain_click)

display(ig_explain_button)
display(ig_explain_output)


Button(button_style='info', description='IG Erklärung generieren', icon='search-plus', style=ButtonStyle())

Output()

### Batch analysis IG

In [ ]:
import ipywidgets as widgets

# --- IG Batch analysis widgets ---
# Filter out baseline caches (.pt tensors) that are not model checkpoints
_filtered_model_files = [f for f in model_files if not f.startswith('baseline_')] or model_files

ig_batch_model_widget = widgets.Dropdown(
    options=_filtered_model_files if _filtered_model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'},
)

ig_batch_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'},
)

ig_batch_baseline_widget = widgets.Dropdown(
    options=['scientific', 'zero', 'mean', 'random', 'min', 'max'],
    value='scientific',
    description='Baseline:',
    style={'description_width': 'initial'},
)

ig_batch_explanation_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'},
)

ig_batch_node_types_widget = widgets.SelectMultiple(
    options=['H', 'C', 'Others'],
    value=('H', 'C'),
    description='Target-Typen:',
    style={'description_width': 'initial'},
)

ig_batch_include_neighbors_widget = widgets.Checkbox(
    value=True,
    description='Context-IG (Nachbarn)',
    indent=False,
)

ig_batch_k_hops_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=4,
    step=1,
    description='k-hops:',
    style={'description_width': 'initial'},
)

ig_batch_max_neighbors_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='max Neighbors/Typ:',
    style={'description_width': 'initial'},
)

ig_batch_n_steps_widget = widgets.IntSlider(
    value=64,
    min=8,
    max=256,
    step=8,
    description='IG steps:',
    style={'description_width': 'initial'},
)

ig_batch_graph_indices_widget = widgets.Text(
    value='',
    description='Graph IDs (optional):',
    placeholder='z.B. 0,2,5 oder 0-10',
    style={'description_width': 'initial'},
)

ig_batch_sample_graphs_widget = widgets.IntSlider(
    value=20,
    min=1,
    max=500,
    step=1,
    description='Erste N Graphen:',
    style={'description_width': 'initial'},
)

ig_batch_output_widget = widgets.Text(
    value='results/ig_batch',
    description='Output Dir:',
    style={'description_width': 'initial'},
)

ig_batch_train_split_widget = widgets.Text(
    value=str(MODELS_DIR / 'graph_split.pkl'),
    description='Train-Split (pkl):',
    style={'description_width': 'initial'},
)

# Layout
controls = widgets.VBox([
    widgets.HBox([ig_batch_model_widget, ig_batch_data_widget, ig_batch_baseline_widget, ig_batch_explanation_type_widget]),
    widgets.HBox([ig_batch_node_types_widget, ig_batch_include_neighbors_widget, ig_batch_k_hops_widget, ig_batch_max_neighbors_widget]),
    widgets.HBox([ig_batch_n_steps_widget, ig_batch_sample_graphs_widget, ig_batch_graph_indices_widget]),
    widgets.HBox([ig_batch_output_widget, ig_batch_train_split_widget]),
])

display(controls)


In [ ]:
from pathlib import Path
import os
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

from scripts.explainer.ig_explainer import batch_explain_nodes_with_ig
from feature_visualization import plot_feature_importance, get_feature_names

def feature_names_for(node_type: str, n_features: int):
    names = list(get_feature_names(node_type) or [])
    if len(names) < n_features:
        names.extend([f"Feature_{i}" for i in range(len(names), n_features)])
    return names[:n_features]

# Printing / Ranking helpers
# ---------------------------------------------------------------------------

def _print_batch_summary(batch_results: dict, graph_indices: list, title: str):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    for nt, result in batch_results.items():
        print(f"\nNode Type (aggregated): {nt}")
        print(f"  Analysierte Graphen: {len(graph_indices)}")

        self_ct = result.get("self_node_count", None)
        ctx_ct  = result.get("ctx_node_count",  None)
        if self_ct is not None or ctx_ct is not None:
            print(f"  Self vectors: {self_ct if self_ct is not None else 0}")
            print(f"  Context vectors (per-target mean): {ctx_ct if ctx_ct is not None else 0}")
        else:
            print(f"  Attribution-Vektoren (node_count): {result.get('node_count', 0)}")

        print(f"  Features: {len(result.get('avg_abs_importance', []))}")

        if result.get("includes_neighbors", False):
            print(
                f"  Context IG: k_hops={result.get('k_hops')}, "
                f"max_neighbors_per_type={result.get('max_neighbors_per_type')}"
            )
            if "context_aggregation" in result:
                print(f"  Context aggregation: {result.get('context_aggregation')}")

        importances_abs = result.get("avg_abs_importance", [])
        if not importances_abs:
            print("  (keine Daten)")
            continue

        print("  Top 5 wichtigste Features (nach avg_abs_importance):")
        feature_names = feature_names_for(nt, len(importances_abs))
        sorted_indices = sorted(
            range(len(importances_abs)),
            key=lambda i: importances_abs[i],
            reverse=True
        )
        for i in range(min(5, len(sorted_indices))):
            idx  = sorted_indices[i]
            name = feature_names[idx]
            print(f"    {name}: {importances_abs[idx]:.6f}")

def _print_joint_top_features(
    batch_results: dict,
    top_k: int | None = None,
    normalize_per_type: bool = False,
):
    rows = []
    global_total = 0.0
    for nt, res in batch_results.items():
        imp = res.get("avg_abs_importance", [])
        if imp:
            global_total += float(np.sum(np.array(imp, dtype=float)))

    for nt, res in batch_results.items():
        imp = res.get("avg_abs_importance", [])
        if not imp:
            continue
        imp = np.array(imp, dtype=float)
        if normalize_per_type:
            denom = float(np.sum(imp))
            if denom < 1e-12:
                continue
            scores = imp / denom
        else:
            scores = imp

        names = feature_names_for(nt, len(scores))
        for i in range(len(scores)):
            rows.append({
                "node_type":    nt,
                "feature_idx":  i,
                "feature_name": names[i],
                "score":        float(scores[i]),
                "global_share": (float(imp[i]) / global_total) if global_total > 1e-12 else 0.0,
            })

    if not rows:
        print("\n[Joint Top Features] Keine Daten vorhanden.")
        return

    rows.sort(key=lambda r: r["score"], reverse=True)
    top_n = len(rows) if (top_k is None or top_k <= 0) else min(top_k, len(rows))
    title_extra = "(alle Features)" if (top_k is None or top_k <= 0) else f"(Top {top_n})"

    print("\n" + "=" * 70)
    print("JOINT FEATURE RANKING (across node types)")
    if normalize_per_type:
        print(f"Ranking basis: per-type normalized avg_abs_importance {title_extra}")
    else:
        print(f"Ranking basis: RAW avg_abs_importance (global magnitude) {title_extra}")
    print("=" * 70)

    for rank, r in enumerate(rows[:top_n], start=1):
        if normalize_per_type:
            print(
                f"{rank:>3}. {r['node_type']}: {r['feature_name']} (idx={r['feature_idx']}) "
                f"| score={r['score']:.6f}"
            )
        else:
            print(
                f"{rank:>3}. {r['node_type']}: {r['feature_name']} (idx={r['feature_idx']}) "
                f"| avg_abs={r['score']:.6f} | global_share={r['global_share']:.4%}"
            )

# ---------------------------------------------------------------------------
# Parsing graph indices (unchanged)
# ---------------------------------------------------------------------------

def _parse_batch_graph_indices(text: str, total_graphs: int):
    text = (text or "").strip()
    if not text:
        return None
    result = []
    seen   = set()
    for token in text.replace(";", ",").split(","):
        token = token.strip()
        if not token:
            continue
        token_clean = token.replace(" ", "")
        if "-" in token_clean:
            parts = token_clean.split("-", 1)
            if len(parts) != 2:
                continue
            try:
                start = int(parts[0]); end = int(parts[1])
            except ValueError:
                continue
            if start > end:
                start, end = end, start
            for idx in range(start, end + 1):
                if 0 <= idx < total_graphs and idx not in seen:
                    seen.add(idx)
                    result.append(idx)
        else:
            try:
                idx = int(token_clean)
            except ValueError:
                continue
            if 0 <= idx < total_graphs and idx not in seen:
                seen.add(idx)
                result.append(idx)
    return sorted(result) if result else None

# ---------------------------------------------------------------------------
# Main runner
# ---------------------------------------------------------------------------

def _latest_ig_batch_file(output_dir: str, target_node_type: str):
    import glob
    pattern = str(Path(output_dir) / f"batch_ig_{target_node_type}*.pt")
    files = glob.glob(pattern)
    if not files:
        return None
    return max(files, key=lambda p: Path(p).stat().st_mtime)


def run_ig_batch_analysis_compute_only():
    data_path_obj   = DATA_DIR / ig_batch_data_widget.value
    config_path_obj = MODELS_DIR / "config.pkl"
    if config_path_obj.exists() and data_path_obj.exists():
        from scripts.explainer.explainer_utils import build_dataset, load_config, load_stats
        config = load_config(str(config_path_obj))
        norm_stats_path_obj = MODELS_DIR / "norm_stats.pkl"
        edge_stats_path_obj = MODELS_DIR / "edge_stats.pkl"
        if norm_stats_path_obj.exists() and edge_stats_path_obj.exists():
            norm_stats, edge_stats = load_stats(str(norm_stats_path_obj), str(edge_stats_path_obj))
            dataset = build_dataset(
                str(data_path_obj),
                config,
                norm_stats=norm_stats,
                edge_stats=edge_stats
            )
            total_graphs = len(dataset)
        else:
            total_graphs = 1000
    else:
        total_graphs = 1000

    custom_graphs = _parse_batch_graph_indices(ig_batch_graph_indices_widget.value, total_graphs)
    if custom_graphs is not None:
        graph_indices = custom_graphs
        preview = graph_indices[:10]
        suffix = "..." if len(graph_indices) > 10 else ""
        print(f"Verarbeite benutzerdefinierte Graphen ({len(graph_indices)}): {preview}{suffix}")
    else:
        sample_graphs = int(ig_batch_sample_graphs_widget.value)
        if sample_graphs <= 0 or sample_graphs >= total_graphs:
            graph_indices = list(range(total_graphs))
            print(f"Verarbeite alle {total_graphs} Graphen")
        else:
            graph_indices = list(range(sample_graphs))
            print(f"Verarbeite erste {sample_graphs} von {total_graphs} Graphen")

    target_node_types = list(ig_batch_node_types_widget.value)
    if not target_node_types:
        print("❌ Fehler: Wähle mindestens einen Node Type aus")
        return None

    include_neighbors = bool(ig_batch_include_neighbors_widget.value)
    k_hops = int(ig_batch_k_hops_widget.value)
    max_neighbors_per_type = int(ig_batch_max_neighbors_widget.value)

    print(f"Target Node Types: {target_node_types}")
    print(f"Integration Steps: {ig_batch_n_steps_widget.value}")
    print(f"Baseline: {ig_batch_baseline_widget.value}")
    print(f"Include neighbors: {include_neighbors}")
    if include_neighbors:
        print(f"  k_hops={k_hops}, max_neighbors_per_type={max_neighbors_per_type}")

    train_split_file = None
    train_split_raw = (ig_batch_train_split_widget.value or "").strip()
    if train_split_raw:
        train_split_path = Path(train_split_raw)
        if not train_split_path.is_absolute():
            train_split_path = PROJECT_ROOT / train_split_path
        train_split_file = str(train_split_path)

    if ig_batch_baseline_widget.value == "scientific":
        if train_split_file:
            print(f"Train-Split-Datei für scientific baseline: {train_split_file}")
        else:
            print("⚠️ Scientific baseline ohne Train-Split-Datei: ig_explainer nutzt 80%-Fallback.")

    model_path      = str(MODELS_DIR / ig_batch_model_widget.value)
    data_path       = str(DATA_DIR   / ig_batch_data_widget.value)
    config_path     = str(MODELS_DIR / "config.pkl")
    norm_stats_path = str(MODELS_DIR / "norm_stats.pkl")
    edge_stats_path = str(MODELS_DIR / "edge_stats.pkl")
    output_dir      = str(PROJECT_ROOT / ig_batch_output_widget.value)

    required_files = [model_path, data_path, config_path, norm_stats_path, edge_stats_path]
    missing_files = [f for f in required_files if not os.path.exists(f)]
    if missing_files:
        print(f"❌ Fehlende Dateien: {missing_files}")
        return None

    results_per_target = {}
    for tgt_type in target_node_types:
        print(f"\n=== RUN für Target-Typ: {tgt_type} (berechnen + speichern) ===")

        tgt_results = batch_explain_nodes_with_ig(
            model_path=model_path,
            data_path=data_path,
            config=config_path,
            norm_stats=norm_stats_path,
            edge_stats=edge_stats_path,
            graph_indices=graph_indices,
            node_type=tgt_type,
            output_dir=output_dir,
            n_steps=int(ig_batch_n_steps_widget.value),
            baseline_type=ig_batch_baseline_widget.value,
            train_split_file=train_split_file,
            include_neighbors=include_neighbors,
            k_hops=k_hops if include_neighbors else 1,
            max_neighbors_per_type=max_neighbors_per_type if include_neighbors else None,
        )

        results_per_target[tgt_type] = tgt_results
        latest_file = _latest_ig_batch_file(output_dir, tgt_type)
        if latest_file:
            print(f"💾 Gespeichert: {latest_file}")
        global_name = (
            f"results_target{tgt_type}_"
            f"{'withNeighbors' if include_neighbors else 'noNeighbors'}"
        )
        globals()[global_name] = tgt_results
        print(f"✅ Gespeichert als global: {global_name}")

        _print_batch_summary(
            tgt_results,
            graph_indices,
            title=(
                f"IG Batch Summary — Target={tgt_type} — "
                f"{'WITH' if include_neighbors else 'NO'} Neighbors (Option B)"
            ),
        )
        _print_joint_top_features(tgt_results, top_k=None, normalize_per_type=False)

        print("[INFO] Visualisierung in separatem Schritt (aus gespeicherten Dateien).")

    global ig_batch_results_per_target, ig_batch_saved_files
    ig_batch_results_per_target = results_per_target
    ig_batch_saved_files = {t: _latest_ig_batch_file(output_dir, t) for t in target_node_types}
    print("\n✅ Alle Target-Runs abgeschlossen und gespeichert.")
    return results_per_target



# ---------------------------------------------------------------------------
# Schritt 1: Berechnen + speichern
# ---------------------------------------------------------------------------

ig_batch_compute_button = widgets.Button(
    description="IG Batch: berechnen + speichern",
    button_style="success",
    icon="save",
)
ig_batch_compute_output = widgets.Output()

def on_ig_batch_compute_click(b):
    with ig_batch_compute_output:
        clear_output()
        run_ig_batch_analysis_compute_only()

ig_batch_compute_button.on_click(on_ig_batch_compute_click)

display(ig_batch_compute_button)
display(ig_batch_compute_output)


Button(button_style='success', description='IG Batch: berechnen + speichern', icon='save', style=ButtonStyle()…

Output()

In [ ]:
import numpy as np
from pathlib import Path
from feature_visualization import get_feature_names

# ── Uni Bonn Corporate Design ────────────────────────────────────────────────
UNI_BLUE    = "#004e9f"
UNI_BLUE_50 = "#7fa6cf"
UNI_BLUE_25 = "#bfd3e7"
UNI_YELLOW  = "#fcba00"
UNI_GREY    = "#909085"
UNI_GREY_25 = "#e3e3e0"
UNI_BLACK   = "#1a1a1a"
UNI_RED     = "#c0392b"
UNI_GREEN   = "#1a7a4a"

# Zusätzliche helle Farben für Nachbarn
UNI_LIGHT_BLUE   = "#a9c4e2"
UNI_LIGHT_YELLOW = "#ffe08a"

# Farben je Rolle
SELF_COLORS = {
    "C": UNI_YELLOW,
    "H": UNI_BLUE,
    "Others": UNI_GREY,
}

NEIGHBOR_COLORS = {
    "C": UNI_LIGHT_YELLOW,
    "H": UNI_LIGHT_BLUE,
    "Others": UNI_GREY_25,
}

def _apply_uni_style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor("#fafaf8")
    ax.figure.patch.set_facecolor("white")
    ax.grid(color=UNI_GREY_25, linewidth=0.8, linestyle="-", alpha=0.9)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=UNI_BLACK, labelsize=9)
    if title:
        ax.set_title(title, color=UNI_BLUE, fontsize=12, fontweight="bold", pad=10, loc="left")
    if xlabel:
        ax.set_xlabel(xlabel, color=UNI_BLACK, fontsize=10, labelpad=6)
    if ylabel:
        ax.set_ylabel(ylabel, color=UNI_BLACK, fontsize=10, labelpad=6)

def _add_value_labels(ax, bars, values, max_abs, fontsize=7.5, signed=False):
    for bar, value in zip(bars, values):
        if abs(value) <= max_abs * 0.01:
            continue
        if signed:
            ha    = "left" if value >= 0 else "right"
            x_pos = bar.get_width() + (max_abs * 0.005 if value >= 0 else -max_abs * 0.005)
        else:
            ha    = "left"
            x_pos = bar.get_width() + max_abs * 0.005
        ax.text(
            x_pos,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            ha=ha,
            va="center",
            fontsize=fontsize,
            color=UNI_BLACK,
        )

def _add_total_labels_right(ax, y_pos, total_values, max_abs, fontsize=7.5):
    offset = max_abs * 0.01 if max_abs > 0 else 0.01
    for y, total in zip(y_pos, total_values):
        if abs(total) <= max_abs * 0.01:
            continue
        ax.text(
            total + offset,
            y,
            f"{total:.3f}",
            ha="left",
            va="center",
            fontsize=fontsize,
            color=UNI_BLACK,
        )

def _safe_name(value: str) -> str:
    cleaned = "".join(ch if (ch.isalnum() or ch in "-_") else "_" for ch in str(value))
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned.strip("_") or "plot"

def _export_figure(fig, export_dir: str | None, stem: str, export_formats=None):
    if not export_dir:
        return
    formats = [str(f).lower().strip() for f in (export_formats or []) if str(f).strip()]
    formats = [f for f in formats if f in {"png", "svg", "pdf"}]
    if not formats:
        return
    out_dir = Path(export_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = _safe_name(stem)
    for fmt in formats:
        out_path = out_dir / f"{stem}.{fmt}"
        fig.savefig(out_path, format=fmt, dpi=300, bbox_inches="tight")
        print(f"💾 Exportiert: {out_path}")

# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Visualisierungen
# ---------------------------------------------------------------------------

def plot_self_ctx_importance(batch_results, show_all=True, top_k=25, export_dir=None, export_formats=None, filename_prefix="ig_batch"):
    """
    Gewünschte Logik:
    - Bei C-Target:
        Diagramm 1: C self + C neighbors
        Diagramm 2: nur H neighbors
    - Bei H-Target:
        Diagramm 1: H self + H neighbors
        Diagramm 2: nur C neighbors

    Farben:
    - C self      = gelb
    - C neighbors = hellgelb
    - H self      = blau
    - H neighbors = hellblau
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib.patches import Patch

    def _pad(arr, n):
        arr = np.array(arr, dtype=float)
        if len(arr) >= n:
            return arr
        out = np.zeros(n, dtype=float)
        out[:len(arr)] = arr
        return out

    # Alle Target-Typen bestimmen, die Self-Importances besitzen
    target_types = []
    for nt, res in batch_results.items():
        self_abs = np.array(res.get("self_avg_abs_importance") or [], dtype=float)
        if self_abs.size > 0:
            target_types.append(nt)

    if not target_types:
        print("⚠️ Keine self_avg_abs_importance gefunden.")
        return

    for target_type in target_types:
        target_res = batch_results[target_type]

        # ------------------------------------------------------------
        # (1) Zieltyp-Diagramm: self + same-type neighbors
        # ------------------------------------------------------------
        self_abs = np.array(target_res.get("self_avg_abs_importance") or [], dtype=float)
        same_ctx_abs = np.array(target_res.get("ctx_avg_abs_importance") or [], dtype=float)

        if self_abs.size == 0 and same_ctx_abs.size == 0:
            continue

        max_len = max(len(self_abs), len(same_ctx_abs))
        self_pad = _pad(self_abs, max_len)
        same_ctx_pad = _pad(same_ctx_abs, max_len)

        combined = self_pad + same_ctx_pad
        order = np.argsort(combined)[::-1]

        if not show_all and top_k is not None and top_k > 0:
            order = order[:top_k]

        names = feature_names_for(target_type, max_len)
        labels = [names[i] for i in order]
        self_vals = self_pad[order]
        same_ctx_vals = same_ctx_pad[order]
        total_vals = combined[order]

        fig_h = max(8, len(order) * 0.35)
        fig, ax = plt.subplots(figsize=(16, fig_h))
        fig.subplots_adjust(left=0.22, right=0.96, top=0.94, bottom=0.06)

        ax.barh(
            np.arange(len(order)),
            self_vals,
            color=SELF_COLORS.get(target_type, UNI_GREY),
            alpha=0.9,
            height=0.65,
            edgecolor="white",
            linewidth=0.4,
            label=f"{target_type} self",
        )

        ax.barh(
            np.arange(len(order)),
            same_ctx_vals,
            left=self_vals,
            color=NEIGHBOR_COLORS.get(target_type, UNI_GREY_25),
            alpha=0.9,
            height=0.65,
            edgecolor="white",
            linewidth=0.4,
            label=f"{target_type} neighbors",
        )

        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()

        _apply_uni_style(
            ax,
            title=f"{target_type} target: self + {target_type} neighbors",
            xlabel="Absolute importance",
            ylabel="Features",
        )

        max_total = total_vals.max() if len(total_vals) and total_vals.max() != 0 else 1.0
        _add_total_labels_right(ax, np.arange(len(order)), total_vals, max_total)
        ax.set_xlim(0, max_total * 1.12)

        ax.legend(
            handles=[
                Patch(facecolor=SELF_COLORS.get(target_type, UNI_GREY), label=f"{target_type} self"),
                Patch(facecolor=NEIGHBOR_COLORS.get(target_type, UNI_GREY_25), label=f"{target_type} neighbors"),
            ],
            title="Contribution",
            fontsize=8.5,
            title_fontsize=9,
            frameon=True,
            framealpha=0.9,
            edgecolor=UNI_GREY_25,
        )

        plt.tight_layout()
        _export_figure(fig, export_dir, f"{filename_prefix}_{target_type}_self_plus_same_neighbors", export_formats)
        plt.show()

        # ------------------------------------------------------------
        # (2) Nachbar-Diagramm: nur der jeweils andere Typ
        # ------------------------------------------------------------
        if target_type == "C":
            other_type = "H"
        elif target_type == "H":
            other_type = "C"
        else:
            other_type = None

        if other_type is None:
            continue

        other_res = batch_results.get(other_type, {})
        other_ctx_abs = np.array(other_res.get("ctx_avg_abs_importance") or [], dtype=float)

        if other_ctx_abs.size == 0:
            continue

        order = np.argsort(other_ctx_abs)[::-1]
        if not show_all and top_k is not None and top_k > 0:
            order = order[:top_k]

        other_names = feature_names_for(other_type, len(other_ctx_abs))
        labels = [other_names[i] for i in order]
        vals = other_ctx_abs[order]

        fig_h = max(8, len(order) * 0.35)
        fig, ax = plt.subplots(figsize=(16, fig_h))
        fig.subplots_adjust(left=0.22, right=0.96, top=0.94, bottom=0.06)

        ax.barh(
            np.arange(len(order)),
            vals,
            color=NEIGHBOR_COLORS.get(other_type, UNI_GREY_25),
            alpha=0.9,
            height=0.65,
            edgecolor="white",
            linewidth=0.4,
        )

        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()

        _apply_uni_style(
            ax,
            title=f"{target_type} target: only {other_type} neighbors",
            xlabel="Absolute neighbor importance",
            ylabel="Features",
        )

        max_val = vals.max() if len(vals) and vals.max() != 0 else 1.0
        _add_total_labels_right(ax, np.arange(len(order)), vals, max_val)
        ax.set_xlim(0, max_val * 1.12)

        ax.legend(
            handles=[
                Patch(facecolor=NEIGHBOR_COLORS.get(other_type, UNI_GREY_25), label=f"{other_type} neighbors")
            ],
            title="Neighbor type",
            fontsize=8.5,
            title_fontsize=9,
            frameon=True,
            framealpha=0.9,
            edgecolor=UNI_GREY_25,
        )

        plt.tight_layout()
        _export_figure(fig, export_dir, f"{filename_prefix}_{target_type}_only_{other_type}_neighbors", export_formats)
        plt.show()

def plot_signed_ig_contributions(batch_results, show_all=True, top_k=25, export_dir=None, export_formats=None, filename_prefix="ig_batch"):
    """Signed IG contributions per node type (positive vs. negative)."""
    import matplotlib.pyplot as plt
    import numpy as np

    available_types = [nt for nt, res in batch_results.items() if len(res.get('avg_importance', [])) > 0]
    if not available_types:
        print('⚠️ No signed IG data found (avg_importance).')
        return

    for nt in [t for t in ['H', 'C', 'Others'] if t in available_types] + [t for t in available_types if t not in ['H', 'C', 'Others']]:
        vals = np.array(batch_results.get(nt, {}).get('avg_importance', []) or [], dtype=float)
        if vals.size == 0:
            continue

        order = np.argsort(np.abs(vals))[::-1]
        if not show_all and top_k is not None and top_k > 0:
            order = order[:top_k]

        names = feature_names_for(nt, len(vals))
        labels = [names[i] for i in order]
        plot_vals = vals[order]
        colors = [UNI_GREEN if v >= 0 else UNI_RED for v in plot_vals]

        fig_h = max(8, len(order) * 0.35)
        fig, ax = plt.subplots(figsize=(16, fig_h))
        fig.subplots_adjust(left=0.22, right=0.96, top=0.94, bottom=0.06)

        bars = ax.barh(np.arange(len(order)), plot_vals, color=colors, alpha=0.9, edgecolor='white', linewidth=0.4, height=0.65)
        ax.axvline(0, color=UNI_BLACK, linewidth=1.0, alpha=0.6)
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()

        _apply_uni_style(
            ax,
            title=f'Signed IG contributions | Node type = {nt}',
            xlabel='Signed contribution (IG attribution)',
            ylabel='Features',
        )

        max_abs = float(np.max(np.abs(plot_vals))) if len(plot_vals) else 1.0
        if max_abs <= 0:
            max_abs = 1.0
        _add_value_labels(ax, bars, plot_vals, max_abs=max_abs, signed=True)
        ax.set_xlim(-max_abs * 1.12, max_abs * 1.12)

        from matplotlib.patches import Patch
        ax.legend(
            handles=[
                Patch(facecolor=UNI_GREEN, label='Positive contribution'),
                Patch(facecolor=UNI_RED, label='Negative contribution'),
            ],
            title='Sign',
            fontsize=8.5,
            title_fontsize=9,
            frameon=True,
            framealpha=0.9,
            edgecolor=UNI_GREY_25,
        )

        plt.tight_layout()
        _export_figure(fig, export_dir, f"{filename_prefix}_{nt}_signed_ig", export_formats)
        plt.show()



def run_ig_batch_analysis_plot_only_from_saved():
    global ig_batch_results_per_target, ig_batch_saved_files
    output_dir = str(PROJECT_ROOT / ig_batch_output_widget.value)
    target_node_types = list(ig_batch_node_types_widget.value)
    if not target_node_types:
        print("❌ Fehler: Wähle mindestens einen Node Type aus")
        return None

    saved_files = {}
    if 'ig_batch_saved_files' in globals() and isinstance(ig_batch_saved_files, dict):
        saved_files.update(ig_batch_saved_files)

    export_formats = list(ig_batch_plot_export_formats_widget.value) if 'ig_batch_plot_export_formats_widget' in globals() else ['png', 'svg']
    do_export = bool(ig_batch_plot_do_export_widget.value) if 'ig_batch_plot_do_export_widget' in globals() else True
    export_dir = str(Path(output_dir) / "plots") if do_export else None

    loaded_results = {}
    for tgt_type in target_node_types:
        fpath = saved_files.get(tgt_type) or _latest_ig_batch_file(output_dir, tgt_type)
        if not fpath:
            print(f"⚠️ Keine gespeicherte Datei für Target={tgt_type} gefunden.")
            continue
        try:
            payload = torch.load(fpath, weights_only=False)
            if not isinstance(payload, dict):
                print(f"⚠️ Ungültige Datei für Target={tgt_type}: {fpath}")
                continue
            loaded_results[tgt_type] = payload
            saved_files[tgt_type] = fpath
            print(f"📂 Geladen ({tgt_type}): {fpath}")
        except Exception as e:
            print(f"⚠️ Laden fehlgeschlagen für {tgt_type}: {e}")

    if not loaded_results:
        print("❌ Keine Ergebnisse zum Plotten geladen.")
        return None

    for tgt_type, tgt_results in loaded_results.items():
        print(f"\n=== PLOT für Target-Typ: {tgt_type} (aus Datei) ===")
        _print_batch_summary(
            tgt_results, [],
            title=(f"IG Batch Summary — Target={tgt_type} — aus gespeicherter Datei"),
        )
        _print_joint_top_features(tgt_results, top_k=None, normalize_per_type=False)
        print("\nErstelle Visualisierung (aus gespeicherten Erklärungen)...")
        try:
            plot_self_ctx_importance(
                tgt_results,
                show_all=True,
                top_k=25,
                export_dir=export_dir,
                export_formats=export_formats,
                filename_prefix=f"ig_batch_target_{tgt_type}",
            )
            plot_signed_ig_contributions(
                tgt_results,
                show_all=True,
                top_k=25,
                export_dir=export_dir,
                export_formats=export_formats,
                filename_prefix=f"ig_batch_target_{tgt_type}",
            )
            print("✅ Visualisierung erfolgreich erstellt!")
        except Exception as e:
            print(f"⚠️ Fehler bei Visualisierung: {e}")

    ig_batch_results_per_target = loaded_results
    ig_batch_saved_files = saved_files
    return loaded_results

# ---------------------------------------------------------------------------
# Widgets / Button wiring
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Schritt 2: Aus gespeicherten Dateien visualisieren
# ---------------------------------------------------------------------------

ig_batch_plot_do_export_widget = widgets.Checkbox(
    value=True,
    description="Plots exportieren",
)
ig_batch_plot_export_formats_widget = widgets.SelectMultiple(
    options=["png", "svg", "pdf"],
    value=("png", "svg"),
    description="Formate",
    layout=widgets.Layout(width="260px", height="95px"),
)

ig_batch_plot_button = widgets.Button(
    description="IG Batch: aus Datei plotten",
    button_style="primary",
    icon="line-chart",
)
ig_batch_plot_output = widgets.Output()

def on_ig_batch_plot_click(b):
    with ig_batch_plot_output:
        clear_output()
        run_ig_batch_analysis_plot_only_from_saved()

ig_batch_plot_button.on_click(on_ig_batch_plot_click)

display(widgets.HBox([ig_batch_plot_do_export_widget, ig_batch_plot_export_formats_widget]))
display(ig_batch_plot_button)
display(ig_batch_plot_output)


Button(button_style='primary', description='IG Batch: aus Datei plotten', icon='line-chart', style=ButtonStyle…

Output()